# Great Britain race-population completeness audit

## Bounded audit question

> Are any Great Britain races that officially produced results missing from Source Version 1 / Database v4?

This is a **database/source-correctness audit**, not a reader-facing study. It follows directly from Great Britain Study 04.

The decisive defect test is:

> **official BHA completed race result -> corresponding Inside Rails source race occurrence**

Scheduled fixture evidence is also useful, but for a different purpose: it helps explain races or fixtures that were abandoned, cancelled, transferred, rescheduled or otherwise changed. A scheduled race or fixture that does not appear in Source Version 1 is **not by itself** evidence of a source-population defect.


## Audit controls and inherited governance

Read before this audit:

- `docs/STUDY_DATABASE_REFERENCE.md`
- `docs/STUDY_DATA_ACCESS.md`
- `docs/RESEARCH_DATA_SOURCE_REGISTER.md`
- `docs/studies/GB_04_RACE_MEETINGS_AND_FIXTURES_CLOSEOUT.md`
- `docs/STUDY_CLOSEOUT_REGISTER.md`

Inherited boundaries:

- Source Version 1 remains immutable and read-only.
- Source admission remains `rowid <> 1`.
- Authorised Source Version 1 race identity remains exact raw `date + course + off`.
- Database v4 is the accepted immutable analytical release.
- Do **not** create a fixture ID, meeting ID or session ID for this audit.
- Do **not** assume `date + racecourse = fixture`.
- Database v4's governed British racecourse identity is the correct geographical bridge for comparison, but it does not identify a physical course/track below racecourse level.


In [4]:
# Audit setup
#
# Establish the exact accepted database and the bounded Great Britain source
# population before acquiring or reconciling any external BHA evidence.

from pathlib import Path

import pandas as pd

from inside_rails.source_sqlite import connect_read_only

PROJECT_ROOT = Path("/home/rob/Documents/inside-rails-horse-racing")
DATABASE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "database"
    / "releases"
    / "inside_rails_v4.sqlite3"
)

assert DATABASE.is_file(), f"Accepted Database v4 not found: {DATABASE}"

with connect_read_only(DATABASE) as connection:
    database_version = connection.execute("PRAGMA user_version").fetchone()[0]
    population_summary = pd.read_sql_query(
        """
        SELECT
            COUNT(*) AS gb_race_occurrences,
            COUNT(DISTINCT source_race_occurrence_code) AS distinct_race_codes,
            MIN(raw_date) AS minimum_source_date,
            MAX(raw_date) AS maximum_source_date,
            COUNT(DISTINCT raw_date) AS source_dates,
            COUNT(DISTINCT candidate_course_label) AS source_course_labels,
            COUNT(DISTINCT racecourse_identity_code) AS governed_racecourses
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        """,
        connection,
    )

assert database_version == 4, f"Expected Database v4, found user_version={database_version}"
assert int(population_summary.loc[0, "gb_race_occurrences"]) == 111_634
assert int(population_summary.loc[0, "distinct_race_codes"]) == 111_634

print(f"Database v{database_version} confirmed.")
population_summary


Database v4 confirmed.


,gb_race_occurrences,distinct_race_codes,minimum_source_date,maximum_source_date,source_dates,source_course_labels,governed_racecourses
0,111634,111634,2015-01-01,2026-05-27,4016,65,61


## Reconciliation model

The audit uses three evidence layers without pretending they are the same thing:

1. **BHA scheduled racing** — what was intended or programmed;
2. **BHA official results** — what actually produced a result;
3. **Source Version 1 / Database v4** — what the third-party source population contains.

The principal completeness denominator is layer 2, not layer 1.

Schedule -> results reconciliation is explanatory. Results -> Database v4 reconciliation is the correctness test.

### Initial classification vocabulary

Keep classifications descriptive until evidence supports something stronger:

- `official_result_matched_source`
- `official_result_candidate_source_match_requires_review`
- `official_result_unmatched_after_reconciliation`
- `scheduled_no_official_result`
- `schedule_changed_or_transferred`
- `external_evidence_unresolved`

Only a fully investigated `official_result_unmatched_after_reconciliation` case can become a genuine `official_result_missing_from_source` defect.


## Authoritative external evidence

Primary source: British Horseracing Authority.

### Results service

`https://www.britishhorseracing.com/racing/results/`

The public results interface exposes realised fixture characteristics including fixture date, fixture type, racecourse, first race and fixture session, and distinguishes fixtures marked **Abandoned** from those for which results can be viewed.

### Fixture lists

`https://www.britishhorseracing.com/racing/fixtures/full-year/`

BHA publishes annual fixture lists. Current material is downloadable, including Excel/PDF formats. The BHA page states that older fixture details can be requested from the Racing Department.

### Evidence boundary

Do not infer a persistent BHA fixture identifier from displayed date/course/session fields. Study 04 already established that the public results description is not a durable fixture key.


## Proposed audit evidence grains

These are **audit artifacts**, not database schema proposals.

### Official result race evidence

One row per BHA race for which the official results service provides a result. Preserve, where exposed:

- result date;
- BHA racecourse name;
- fixture/session context;
- race time / race ordering information;
- race name or description;
- stable public result URL or other BHA reference if available;
- enough result detail to disambiguate a candidate match where necessary;
- retrieval timestamp and source locator.

### Scheduled fixture evidence

One row per published BHA scheduled fixture observation, preserving the fields actually supplied by the relevant annual list. Do not invent missing race-programme detail.

The annual fixture list can support schedule/result context even if it is not sufficient to reconstruct every historical scheduled race.


## Matching strategy

Do not match using raw source `race_id`.

Candidate matching should start with the strongest common observable facts available on both sides, normally:

- official/result date;
- governed racecourse identity after explicit BHA-name -> Inside Rails racecourse reconciliation;
- race time / source raw `off` where semantically comparable.

If that is not unique or a time has changed, use additional race evidence such as race name and runner/result signatures to resolve the case.

A failed first-pass join is an **investigation queue**, not proof of a missing race. Transfers, racecourse naming differences, time amendments and programme changes must be exhausted first.


In [5]:
# Database-side pilot population
#
# Use the final Source Version 1 date as a small concrete slice while we
# establish the reproducible BHA result-acquisition route. This does not yet
# attempt any external match.

pilot_date = "2026-05-27"

with connect_read_only(DATABASE) as connection:
    db_pilot = pd.read_sql_query(
        """
        SELECT
            source_race_occurrence_code,
            raw_date,
            raw_course,
            raw_off,
            candidate_course_label,
            governed_racecourse_name,
            racecourse_identity_code,
            race_name_raw,
            governed_runner_count
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        WHERE raw_date = ?
        ORDER BY governed_racecourse_name, raw_off, source_race_occurrence_code
        """,
        connection,
        params=(pilot_date,),
    )

print(f"Database v4 races on {pilot_date}: {len(db_pilot):,}")
db_pilot


Database v4 races on 2026-05-27: 34


,source_race_occurrence_code,raw_date,raw_course,raw_off,candidate_course_label,governed_racecourse_name,racecourse_identity_code,race_name_raw,governed_runner_count
0,race:77b5dbbbfdee69d4d92a5826:000189000,2026-05-27,Beverley,14:15,Beverley,Beverley,rc:gb:beverley,Happy Birthday Joe McCabe Claiming Stakes,7
1,race:77b5dbbbfdee69d4d92a5826:000188996,2026-05-27,Beverley,14:45,Beverley,Beverley,rc:gb:beverley,Tigers Trust Restricted Novice Stakes (For hor...,8
2,race:77b5dbbbfdee69d4d92a5826:000188997,2026-05-27,Beverley,15:15,Beverley,Beverley,rc:gb:beverley,Ward Homes Yorkshire 10th Anniversary Handicap,11
3,race:77b5dbbbfdee69d4d92a5826:000188995,2026-05-27,Beverley,15:45,Beverley,Beverley,rc:gb:beverley,Dr Eddie Moll Handicap,7
4,race:77b5dbbbfdee69d4d92a5826:000188999,2026-05-27,Beverley,16:15,Beverley,Beverley,rc:gb:beverley,Connexins Full Fibre For All Handicap,7
5,race:77b5dbbbfdee69d4d92a5826:000188998,2026-05-27,Beverley,16:50,Beverley,Beverley,rc:gb:beverley,Up The Tigers Handicap,6
6,race:77b5dbbbfdee69d4d92a5826:000188994,2026-05-27,Beverley,17:20,Beverley,Beverley,rc:gb:beverley,Racing Again This Saturday Apprentice Handicap,9
7,race:77b5dbbbfdee69d4d92a5826:000189006,2026-05-27,Cartmel,18:08,Cartmel,Cartmel,rc:gb:cartmel,William Hill Maiden Hurdle (GBB Race),7
8,race:77b5dbbbfdee69d4d92a5826:000189003,2026-05-27,Cartmel,18:38,Cartmel,Cartmel,rc:gb:cartmel,In Memory Of Jimmy Latham Selling Handicap Hurdle,13
9,race:77b5dbbbfdee69d4d92a5826:000189001,2026-05-27,Cartmel,19:08,Cartmel,Cartmel,rc:gb:cartmel,Traffic Management Handicap Hurdle,9


## Question 1 — Can the BHA completed-result population be acquired reproducibly?

### Why this comes first

A source-completeness percentage is meaningless unless the external denominator itself is demonstrably complete for the requested period.

### Smallest next test

Before writing a bulk collector:

1. inspect the BHA results interface and its underlying request/response structure;
2. retrieve one bounded date with known realised racing;
3. prove that every fixture and individual race displayed for that date can be represented without relying on an inferred fixture identity;
4. compare that single-date official result population with `db_pilot`;
5. only then generalise the acquisition method across the Source Version 1 period.

Do not bulk-scrape first and discover later that pagination, abandoned fixtures or historical limits were misunderstood.


## 2026 bounded acquisition case

The first generalisation step is now deliberately **the whole of 2026**, not the whole 2015-2026 source period.

This gives the audit a live-year case in which the original published plan can be compared with later fixture state and realised official results before the acquisition method is applied historically.

### Keep three 2026 evidence layers separate

1. **Original published 2026 fixture plan** — the BHA annual fixture list as originally published.
2. **Observed 2026 fixture state** — later BHA observations showing what is scheduled, altered, transferred, cancelled or abandoned at retrieval time.
3. **Official 2026 race results** — individual races that actually produced official results.

The original annual plan must not be overwritten by later observations. Fixture evidence is mutable; each observation requires provenance and retrieval time.

### Source-completeness boundary

The actual Source Version 1 / Database v4 completeness test remains bounded to **2026-01-01 through 2026-05-27**, because Source Version 1 ends on 2026-05-27.

Official results after 2026-05-27 are useful for validating the BHA acquisition method, but they are outside Source Version 1 coverage and cannot be classified as source omissions. Future scheduled fixtures are planning observations only and must never enter the completed-race denominator.

### Preserve the raw BHA documents

Static BHA 2026 evidence is retained unchanged under `data/external/bha/gb_race_population_completeness/2026/`. The bounded downloader `scripts/download_bha_2026_audit_evidence.py` records source URL, retrieval timestamp, byte size and SHA-256 in `manifest.json`. The retained set includes the 2026 Fixture List PDF/Excel, Headline Measures PDF and January-May 2026 Racing Data Packs.

### Revised progression

1. Prove the results acquisition route on **2026-05-27**.
2. Generalise the same route to all of **2026**.
3. Reconcile original plan -> observed fixture state -> official results without treating those layers as interchangeable.
4. Compare official results through 2026-05-27 with Database v4.
5. Only after the 2026 method is understood and reproducible, extend historical result acquisition across 2015-2025.


### Step 1 — Inspect the original 2026 BHA fixture workbook

Before parsing the annual fixture list, inspect the retained original **inside this notebook**.

The purpose of this step is deliberately narrow:

- confirm the workbook and worksheets we actually received from BHA;
- inspect the row/column structure of `List - Full Year`;
- identify the supplied header fields and the apparent row grain;
- **do not yet assume** that one spreadsheet row is a governed fixture identity;
- **do not modify** the retained XLSX.

The workbook is planning evidence. Even if its list sheet proves to be one row per published fixture observation, it remains the **original plan**, not the completed-race denominator.


In [6]:
# Inspect the retained BHA 2026 fixture workbook in-place and read-only.
#
# We intentionally use only Python's standard library here. An XLSX file is a
# ZIP archive of XML documents, so this lets the audit inspect the source
# without introducing a new project dependency merely for this first look.
#
# This cell is exploratory evidence inspection, not a parser. We are looking
# for the workbook's actual structure and displayed values before deciding
# what a governed extraction should look like.

import re
import zipfile
import xml.etree.ElementTree as ET

BHA_2026_EVIDENCE_DIR = (
    PROJECT_ROOT
    / "data"
    / "external"
    / "bha"
    / "gb_race_population_completeness"
    / "2026"
)
FIXTURE_WORKBOOK = BHA_2026_EVIDENCE_DIR / "2026_Fixture_List.xlsx"

assert FIXTURE_WORKBOOK.is_file(), f"BHA fixture workbook not found: {FIXTURE_WORKBOOK}"

# XLSX namespace constants. Keeping them explicit makes the XML traversal
# auditable rather than hiding workbook interpretation behind a convenience
# library.
MAIN_NS = "http://schemas.openxmlformats.org/spreadsheetml/2006/main"
REL_NS = "http://schemas.openxmlformats.org/package/2006/relationships"
DOC_REL_NS = "http://schemas.openxmlformats.org/officeDocument/2006/relationships"
NS = {"x": MAIN_NS}
RELATIONSHIP_NS = {"r": REL_NS}

with zipfile.ZipFile(FIXTURE_WORKBOOK) as archive:
    workbook_xml = ET.fromstring(archive.read("xl/workbook.xml"))
    workbook_relationships = ET.fromstring(
        archive.read("xl/_rels/workbook.xml.rels")
    )

    # Map workbook relationship IDs to the worksheet XML files they point to.
    relationship_targets = {
        relationship.attrib["Id"]: relationship.attrib["Target"]
        for relationship in workbook_relationships.findall(
            "r:Relationship", RELATIONSHIP_NS
        )
    }

    sheet_targets = {}
    for sheet in workbook_xml.find("x:sheets", NS):
        relationship_id = sheet.attrib[f"{{{DOC_REL_NS}}}id"]
        target = relationship_targets[relationship_id]
        sheet_targets[sheet.attrib["name"]] = f"xl/{target}"

    print("Workbook sheets:")
    for name, target in sheet_targets.items():
        print(f"  - {name}: {target}")

    # The workbook exposes several presentation/summary sheets. We inspect
    # 'List - Full Year' because its name suggests the row-level annual list,
    # but we still test its contents rather than assuming its grain.
    LIST_SHEET_NAME = "List - Full Year"
    assert LIST_SHEET_NAME in sheet_targets, (
        f"Expected worksheet {LIST_SHEET_NAME!r}; found {list(sheet_targets)}"
    )

    # Shared strings are stored separately in many XLSX workbooks. Build the
    # lookup only if the workbook actually contains that part.
    shared_strings = []
    if "xl/sharedStrings.xml" in archive.namelist():
        shared_strings_xml = ET.fromstring(archive.read("xl/sharedStrings.xml"))
        for item in shared_strings_xml.findall("x:si", NS):
            shared_strings.append(
                "".join(text.text or "" for text in item.iter(f"{{{MAIN_NS}}}t"))
            )

    sheet_xml = ET.fromstring(archive.read(sheet_targets[LIST_SHEET_NAME]))
    dimension = sheet_xml.find("x:dimension", NS)
    print(
        "\nDeclared used range:",
        dimension.attrib.get("ref") if dimension is not None else None,
    )

    def cell_value(cell):
        # Return the stored/displayed value for a worksheet cell.
        cell_type = cell.attrib.get("t")
        value_node = cell.find("x:v", NS)

        if cell_type == "inlineStr":
            inline = cell.find("x:is", NS)
            if inline is None:
                return None
            return "".join(
                text.text or "" for text in inline.iter(f"{{{MAIN_NS}}}t")
            )

        if value_node is None:
            return None

        value = value_node.text
        if cell_type == "s":
            return shared_strings[int(value)]
        if cell_type == "b":
            return value == "1"
        return value

    def column_number(cell_reference):
        # Convert an Excel reference such as C12 to its 1-based column number.
        letters = re.match(r"[A-Z]+", cell_reference).group(0)
        number = 0
        for letter in letters:
            number = number * 26 + (ord(letter) - ord("A") + 1)
        return number

    # Preserve blank cells between populated cells so the preview reflects the
    # actual worksheet column positions rather than collapsing them.
    preview_rows = []
    sheet_data = sheet_xml.find("x:sheetData", NS)
    for row in list(sheet_data)[:15]:
        values_by_column = {
            column_number(cell.attrib["r"]): cell_value(cell)
            for cell in row.findall("x:c", NS)
        }
        last_column = max(values_by_column, default=0)
        preview_rows.append(
            [values_by_column.get(column) for column in range(1, last_column + 1)]
        )

# Pad rows to a common width only for display. No interpretation or cleaning
# has happened yet.
preview_width = max(map(len, preview_rows), default=0)
fixture_workbook_preview = pd.DataFrame(
    [row + [None] * (preview_width - len(row)) for row in preview_rows]
)

print("\nFirst 15 stored rows from 'List - Full Year':")
fixture_workbook_preview


Workbook sheets:
  - List - Full Year: xl/worksheets/sheet1.xml
  - Grid - Full Year: xl/worksheets/sheet2.xml
  - Premier Fixture List: xl/worksheets/sheet3.xml
  - Fixture Numbers: xl/worksheets/sheet4.xml

Declared used range: A1:I1459

First 15 stored rows from 'List - Full Year':


,0,1,2,3,4,5,6,7,8
0,Date,Weekday,Course,Time,CourseGroup,Region,Code,Surface,Type
1,46023,Thursday,Windsor,Afternoon,Arena Racing Corporation Limited,South,Jump,Turf,National/BHA
2,46023,Thursday,Southwell,Afternoon,Arena Racing Corporation Limited,Midlands,Flat,AWT,National/BHA
3,46023,Thursday,Musselburgh,Afternoon,Chester Race Company Limited,North,Jump,Turf,National/BHA
4,46023,Thursday,Catterick Bridge,Afternoon,Independent,North,Jump,Turf,Racecourse/Normal
5,46023,Thursday,Cheltenham,Afternoon,Jockey Club Racecourses Limited,Midlands,Jump,Turf,Racecourse/Normal
6,46023,Thursday,Exeter,Afternoon,Jockey Club Racecourses Limited,South,Jump,Turf,Racecourse/Normal
7,46023,Thursday,Newcastle,Afternoon,Arena Racing Corporation Limited,North,Flat,AWT,Racecourse/Normal
8,46024,Friday,Ayr,Afternoon,Independent,North,Jump,Turf,Racecourse/Normal
9,46024,Friday,Fakenham,Afternoon,Independent,Midlands,Jump,Turf,Racecourse/Normal


#### Interpretation checkpoint

Stop here after running the cell.

The output should tell us what BHA actually supplied: worksheet names, the declared used range, the header structure and the first fixture rows. **Only then** should the next cell turn the sheet into a structured fixture-plan table.

In particular, we should not write a parser that silently treats a row as a fixture until the displayed workbook structure supports that interpretation.


### Step 2 — Structure and validate the original 2026 fixture-plan rows

The `List - Full Year` worksheet contains 1,459 stored rows: one header row plus 1,458 data rows.

That exactly matches the BHA's published total of 1,458 fixtures for 2026, so the sheet is a strong candidate for the original published fixture-plan population.

Before using it as such, this step checks its internal structure rather than assuming it is clean:

- convert the stored Excel date serials into calendar dates;
- verify the supplied weekday agrees with the converted date;
- check all nine supplied fields for missing values;
- check for exact duplicate rows;
- inspect repeated `date + course` combinations;
- preserve every BHA field exactly as supplied alongside the converted date.

A spreadsheet row is treated here only as a **published fixture-plan observation**. It is not being promoted to a persistent fixture identity, and `date + course` is not assumed to identify a fixture.

In [8]:
# Read and validate the BHA's original published 2026 fixture-plan rows.
#
# We already inspected the workbook structure in the previous cell and found:
#   - worksheet: "List - Full Year"
#   - range: A1:I1459
#   - 1 header row + 1,458 data rows
#
# BHA's published total for the 2026 Fixture List is also 1,458 fixtures.
#
# That strongly suggests this sheet is the row-level representation of the
# original published fixture plan. This cell now tests that interpretation.
#
# IMPORTANT:
# A row here is only a "published fixture-plan observation".
# We are NOT creating a permanent fixture identity.
# We are also NOT assuming date + racecourse uniquely identifies a fixture.


from collections import Counter, defaultdict
from datetime import datetime, timedelta
import zipfile
import xml.etree.ElementTree as ET


# Re-use the workbook path established in the previous notebook cell.
assert FIXTURE_WORKBOOK.is_file(), (
    f"BHA fixture workbook not found: {FIXTURE_WORKBOOK}"
)


# Open the XLSX read-only.
#
# XLSX files are ZIP archives containing XML files, so we can inspect the
# original source directly without installing another spreadsheet package.
with zipfile.ZipFile(FIXTURE_WORKBOOK) as archive:

    # Many Excel text values are stored once in sharedStrings.xml and worksheet
    # cells then point to them by numeric index.
    #
    # Build that lookup table first so "2" can become "Windsor", for example,
    # when the worksheet says that cell uses a shared string.
    shared_strings = []

    if "xl/sharedStrings.xml" in archive.namelist():

        shared_strings_xml = ET.fromstring(
            archive.read("xl/sharedStrings.xml")
        )

        for item in shared_strings_xml.findall("x:si", NS):

            # One Excel string can technically contain several text fragments.
            # Join them so we recover the complete displayed cell value.
            text_value = "".join(
                text.text or ""
                for text in item.iter(f"{{{MAIN_NS}}}t")
            )

            shared_strings.append(text_value)


    # Step 1 already established which XML file belongs to "List - Full Year".
    #
    # Read that exact worksheet rather than assuming that sheet1.xml will
    # always be the list sheet.
    sheet_xml = ET.fromstring(
        archive.read(sheet_targets["List - Full Year"])
    )

    sheet_data = sheet_xml.find("x:sheetData", NS)

    assert sheet_data is not None, (
        "'List - Full Year' contains no worksheet data."
    )


    def read_cell_value(cell):
        """
        Recover the stored value from one XLSX cell.

        We deliberately do not interpret dates here. At this stage we only
        recover what Excel stored. Date interpretation happens explicitly
        later so that transformation remains visible in the audit.
        """

        cell_type = cell.attrib.get("t")
        value_node = cell.find("x:v", NS)

        # Some strings are stored directly inside the cell instead of through
        # sharedStrings.xml.
        if cell_type == "inlineStr":
            inline = cell.find("x:is", NS)

            if inline is None:
                return None

            return "".join(
                text.text or ""
                for text in inline.iter(f"{{{MAIN_NS}}}t")
            )

        # A genuinely blank cell has no value element.
        if value_node is None:
            return None

        raw_value = value_node.text

        # Shared-string cells contain an integer lookup position.
        if cell_type == "s":
            return shared_strings[int(raw_value)]

        # Excel booleans are stored as 0/1.
        if cell_type == "b":
            return raw_value == "1"

        # Numeric values, including Excel date serials, remain as raw strings
        # for now.
        return raw_value


    def read_row(row):
        """
        Read columns A:I while preserving blank column positions.

        This matters because simply iterating over populated cells could cause
        later values to shift left if a source field were blank.
        """

        values_by_column = {}

        for cell in row.findall("x:c", NS):

            # column_number() came from Step 1 and turns e.g. "C42" into 3.
            column = column_number(cell.attrib["r"])

            values_by_column[column] = read_cell_value(cell)

        # The inspected source has exactly nine fields, A through I.
        return [
            values_by_column.get(column)
            for column in range(1, 10)
        ]


    # Recover every stored row from the row-level worksheet.
    stored_rows = [
        read_row(row)
        for row in sheet_data.findall("x:row", NS)
    ]


# The first worksheet row is the field header; everything after it is a
# published plan observation.
assert stored_rows, "The BHA fixture-list worksheet is unexpectedly empty."

headers = stored_rows[0]
data_rows = stored_rows[1:]


# These are the exact headers observed during Step 1.
#
# If BHA ever changes the workbook layout, fail here rather than silently
# treating a different structure as equivalent evidence.
expected_headers = [
    "Date",
    "Weekday",
    "Course",
    "Time",
    "CourseGroup",
    "Region",
    "Code",
    "Surface",
    "Type",
]

assert headers == expected_headers, (
    "Unexpected BHA fixture-list structure.\n"
    f"Expected: {expected_headers}\n"
    f"Found:    {headers}"
)


# This reconciles the workbook's row population to the BHA's published
# headline count.
assert len(data_rows) == 1_458, (
    f"Expected 1,458 published rows; found {len(data_rows):,}."
)


# Give the source fields audit-friendly names.
#
# 'bha_date_serial' deliberately preserves the original Excel serial rather
# than replacing it with our derived calendar date.
source_field_names = [
    "bha_date_serial",
    "weekday",
    "course",
    "time",
    "course_group",
    "region",
    "code",
    "surface",
    "type",
]


# Excel's normal serial-date system can be converted using 1899-12-30 as the
# origin. For example, serial 46023 should resolve to 2026-01-01.
EXCEL_DATE_ORIGIN = datetime(1899, 12, 30)


# Convert each worksheet row into a named observation.
original_2026_fixture_plan = []

for values in data_rows:

    # Preserve all nine original BHA values exactly as read.
    observation = dict(zip(source_field_names, values))

    raw_serial = observation["bha_date_serial"]

    # Create a separate derived calendar date.
    #
    # This does not overwrite the source serial, so the transformation remains
    # traceable.
    if raw_serial is None:
        fixture_date = None
    else:
        fixture_date = (
            EXCEL_DATE_ORIGIN
            + timedelta(days=float(raw_serial))
        ).date()

    observation["fixture_date"] = fixture_date

    # Independently calculate the weekday.
    #
    # This gives us a useful internal consistency check: if BHA says "Thursday"
    # and the converted date says "Friday", either our date conversion or the
    # source itself needs investigation.
    observation["derived_weekday"] = (
        fixture_date.strftime("%A")
        if fixture_date is not None
        else None
    )

    original_2026_fixture_plan.append(observation)


# Find any date/weekday disagreement.
weekday_mismatches = [
    row
    for row in original_2026_fixture_plan
    if row["weekday"] != row["derived_weekday"]
]


# Count blanks in every field supplied by BHA.
#
# A blank is not automatically an error. We just need to know whether any
# fields are incomplete before relying on them later for reconciliation.
missing_values = {}

for field in source_field_names:

    missing_values[field] = sum(
        row[field] is None or row[field] == ""
        for row in original_2026_fixture_plan
    )


# Check whether the workbook contains exact duplicate source rows.
#
# If there are duplicates, the headline total of 1,458 rows would not
# necessarily mean 1,458 different published observations.
source_row_keys = [
    tuple(row[field] for field in source_field_names)
    for row in original_2026_fixture_plan
]

source_row_counts = Counter(source_row_keys)

duplicate_keys = {
    key
    for key, count in source_row_counts.items()
    if count > 1
}

exact_duplicate_rows = [
    row
    for row in original_2026_fixture_plan
    if tuple(row[field] for field in source_field_names)
    in duplicate_keys
]


# Group rows by date + course.
#
# This is a TEST, not an identity rule.
#
# Study 04 specifically established that date + racecourse must not simply be
# assumed to equal one fixture. If BHA's own fixture list contains repeated
# date/course combinations, that is direct evidence supporting that decision.
date_course_groups = defaultdict(list)

for row in original_2026_fixture_plan:

    key = (
        row["fixture_date"],
        row["course"],
    )

    date_course_groups[key].append(row)


# Keep only date/course combinations represented by more than one published
# row so we can inspect why they differ.
repeated_date_course = {
    key: rows
    for key, rows in date_course_groups.items()
    if len(rows) > 1
}


# Print the high-level validation results first.
print("Original 2026 fixture-plan validation")
print("-------------------------------------")

print(
    "Published fixture-plan rows:",
    f"{len(original_2026_fixture_plan):,}",
)

print(
    "Distinct calendar dates:",
    len({
        row["fixture_date"]
        for row in original_2026_fixture_plan
    }),
)

print(
    "Distinct BHA course labels:",
    len({
        row["course"]
        for row in original_2026_fixture_plan
    }),
)

print(
    "Weekday/date mismatches:",
    len(weekday_mismatches),
)

print(
    "Exact duplicate rows:",
    len(exact_duplicate_rows),
)

print(
    "Repeated date + course combinations:",
    len(repeated_date_course),
)


# Show source-field completeness separately so it remains easy to inspect.
print("\nMissing values by BHA field")
print("---------------------------")

for field, count in missing_values.items():
    print(f"{field:20} {count:,}")


# Show the repeated date/course observations without collapsing them.
#
# Time, code, surface and type may explain why BHA published two separate rows
# on the same date at the same course.
print("\nRepeated date + course combinations")
print("-----------------------------------")

if not repeated_date_course:

    print("None")

else:

    for (fixture_date, course), rows in sorted(
        repeated_date_course.items()
    ):

        print(
            f"\n{fixture_date} | {course} | "
            f"{len(rows)} published rows"
        )

        for row in rows:

            print(
                "   ",
                {
                    "time": row["time"],
                    "code": row["code"],
                    "surface": row["surface"],
                    "type": row["type"],
                },
            )


# Only print detailed warnings if the validation actually found something that
# requires investigation.
if weekday_mismatches:

    print("\nWARNING — weekday/date mismatches:")

    for row in weekday_mismatches:
        print(row)


if exact_duplicate_rows:

    print("\nWARNING — exact duplicate source rows:")

    for row in exact_duplicate_rows:
        print(row)


# Finally, show a small sample of the structured observations so we can verify
# visually that the raw BHA fields and our derived date coexist as intended.
print("\nFirst 10 structured observations")
print("--------------------------------")

for row in original_2026_fixture_plan[:10]:
    print(row)

Original 2026 fixture-plan validation
-------------------------------------
Published fixture-plan rows: 1,458
Distinct calendar dates: 362
Distinct BHA course labels: 59
Weekday/date mismatches: 1
Exact duplicate rows: 0
Repeated date + course combinations: 0

Missing values by BHA field
---------------------------
bha_date_serial      0
weekday              0
course               0
time                 0
course_group         0
region               0
code                 0
surface              0
type                 0

Repeated date + course combinations
-----------------------------------
None

WARNING — weekday/date mismatches:
{'bha_date_serial': '46310', 'weekday': 'Saturday', 'course': 'Market Rasen', 'time': 'Afternoon', 'course_group': 'Jockey Club Racecourses Limited', 'region': 'Midlands', 'code': 'Jump', 'surface': 'Turf', 'type': 'Racecourse/Normal', 'fixture_date': datetime.date(2026, 10, 15), 'derived_weekday': 'Thursday'}

First 10 structured observations
---------------

### Fixture-plan validation conclusion

The retained 2026 BHA fixture-list workbook contains:

- **1,458 published fixture-plan rows**;
- **362 calendar dates**;
- **59 BHA course labels**;
- **no missing values** in the nine supplied fields;
- **no exact duplicate rows**;
- **no repeated `date + course` combinations**.

One internal inconsistency was detected:

- Market Rasen has Excel date serial `46310`, which converts to **Thursday 15 October 2026**;
- the supplied `Weekday` field says **Saturday**.

BHA separately published a fixture-change notice stating that Market Rasen's fixture originally scheduled for Thursday 15 October was moved to Saturday 17 October 2026.

That external notice explains the relevant fixture history, but it does **not** establish why this particular workbook row contains the old date and the Saturday weekday. The workbook inconsistency therefore remains recorded without inferring its cause.

The fixture-list evidence remains planning/context evidence. It is **not** the denominator for the source-completeness test.

## Step 3 — Probe the BHA official-results acquisition route

The next question is narrower:

> Can we reproducibly acquire the BHA official completed-result population for 27 May 2026?

Before collecting anything in bulk, inspect one candidate BHA data route and establish:

1. whether it is currently reachable;
2. what JSON structure it returns;
3. what fixture identifiers and dates it exposes;
4. how pagination works;
5. whether abandoned fixtures are represented explicitly.

No result population will be accepted until the acquisition route itself is understood.

In [9]:
# Probe the candidate data route used behind the BHA racing website.
#
# WHY THIS CELL EXISTS
# --------------------
# The public BHA Results page is rendered dynamically, so the ordinary HTML
# does not contain the individual fixture/result records we need for this
# completeness audit.
#
# Independent technical evidence points to this JSON service as infrastructure
# used by the BHA website:
#
#     https://api09.horseracing.software/bha/v1/fixtures
#
# That is NOT the same thing as having official BHA API documentation.
#
# We therefore treat the endpoint only as a candidate acquisition route and
# perform the smallest possible diagnostic request before relying on it.
#
# IMPORTANT:
#   - this cell does not bulk-download anything;
#   - it does not assume the response schema;
#   - it does not infer fixture identity;
#   - it does not yet compare anything with Database v4.


import json
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen


# Candidate backend root observed in technical material relating to the BHA
# website. Keep it explicit so the provenance of the request is obvious.
BHA_CANDIDATE_API_ROOT = (
    "https://api09.horseracing.software/bha/v1/fixtures"
)


# Request ONE page only.
#
# We deliberately start with page 1 because the purpose is to understand the
# response structure and pagination metadata, not to find 27 May yet.
probe_url = f"{BHA_CANDIDATE_API_ROOT}?page=1"


# The service has previously expected requests to identify the BHA website as
# their origin.
#
# Supplying a normal browser User-Agent also makes this diagnostic request
# resemble the type of request the public BHA site itself would make.
probe_headers = {
    "Accept": "application/json",
    "Origin": "https://www.britishhorseracing.com",
    "User-Agent": (
        "Mozilla/5.0 "
        "(X11; Ubuntu; Linux x86_64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/120 Safari/537.36"
    ),
}


request = Request(
    probe_url,
    headers=probe_headers,
    method="GET",
)


# Keep the raw response variables separate from the parsed JSON.
#
# That way, if something unexpected comes back — HTML, an error document,
# Cloudflare-style protection, etc. — we can see what actually happened rather
# than immediately failing inside json.loads().
response_status = None
response_content_type = None
response_body = None
probe_payload = None


try:

    with urlopen(request, timeout=30) as response:

        response_status = response.status

        response_content_type = response.headers.get(
            "Content-Type"
        )

        response_body = response.read()


except HTTPError as error:

    # HTTP errors still have useful status codes and often useful response
    # bodies. Preserve both for diagnosis.
    response_status = error.code

    response_content_type = error.headers.get(
        "Content-Type"
    )

    response_body = error.read()


except URLError as error:

    # Network/DNS/TLS failures are different from the server deliberately
    # returning an HTTP error, so report them separately.
    print("Network-level failure while probing candidate BHA data route:")
    print(error)

    response_body = None


print("Candidate BHA data-route probe")
print("------------------------------")
print("URL:", probe_url)
print("HTTP status:", response_status)
print("Content-Type:", response_content_type)

if response_body is not None:
    print("Response bytes:", f"{len(response_body):,}")


# Only attempt JSON parsing if we actually received a response body.
if response_body:

    try:

        # Decode explicitly so an encoding or non-JSON response cannot be
        # mistaken for valid structured evidence.
        response_text = response_body.decode("utf-8")

        probe_payload = json.loads(response_text)

        print("\nJSON parsing: SUCCESS")

    except (UnicodeDecodeError, json.JSONDecodeError) as error:

        print("\nJSON parsing: FAILED")
        print(type(error).__name__, error)

        # Show only the beginning of an unexpected response.
        #
        # Avoid dumping a potentially huge HTML/error document into the
        # notebook.
        print("\nFirst 1,000 response characters:")
        print(
            response_body[:1000].decode(
                "utf-8",
                errors="replace",
            )
        )


# If valid JSON was returned, inspect its structure WITHOUT assuming that it
# contains fields called "data", "page", "total", etc.
if probe_payload is not None:

    print("\nTop-level JSON type:")
    print(type(probe_payload).__name__)

    if isinstance(probe_payload, dict):

        print("\nTop-level keys:")
        print(sorted(probe_payload.keys()))

        # Print scalar top-level values because these often contain pagination
        # information such as current page, final page or record counts.
        #
        # Do not print nested lists/dictionaries here; they may be large.
        print("\nTop-level scalar metadata:")

        scalar_metadata_found = False

        for key, value in probe_payload.items():

            if not isinstance(value, (list, dict)):

                print(f"{key}: {value!r}")

                scalar_metadata_found = True

        if not scalar_metadata_found:
            print("No scalar top-level metadata found.")


        # Many APIs use a "data" list, but we test for it explicitly rather
        # than assuming it exists.
        candidate_rows = probe_payload.get("data")

        if isinstance(candidate_rows, list):

            print(
                "\nRows in top-level 'data' list:",
                len(candidate_rows),
            )

            # Inspect only the first three records.
            #
            # At this point the important evidence is the schema: which fields
            # exist and what sorts of values they contain.
            for index, row in enumerate(
                candidate_rows[:3],
                start=1,
            ):

                print(f"\nRecord {index}:")

                if isinstance(row, dict):

                    print("Fields:")
                    print(sorted(row.keys()))

                    print("Values:")
                    print(row)

                else:

                    print(
                        "Unexpected record type:",
                        type(row).__name__,
                    )
                    print(row)

        else:

            print(
                "\nNo top-level list called 'data' was returned."
            )

    else:

        # A valid JSON response need not necessarily be an object.
        # Show a bounded representation if the service returns another type.
        print("\nBounded JSON preview:")
        print(repr(probe_payload)[:2000])

Candidate BHA data-route probe
------------------------------
URL: https://api09.horseracing.software/bha/v1/fixtures?page=1
HTTP status: 401
Content-Type: application/json
Response bytes: 30

JSON parsing: SUCCESS

Top-level JSON type:
dict

Top-level keys:
['message']

Top-level scalar metadata:
message: 'Unauthenticated.'

No top-level list called 'data' was returned.


### Candidate API probe conclusion

The first candidate backend request returned:

- HTTP `401`;
- JSON content;
- message: `Unauthenticated.`

This establishes that the previously identified endpoint is currently reachable but cannot be used through the unauthenticated request attempted here.

No authentication mechanism will be guessed or inferred.

The next step is therefore to inspect the **current public BHA Results page itself** and identify the JavaScript resources it loads. This keeps the investigation anchored to the live public BHA service rather than assuming that an older technical description still reflects the site's current implementation.

In [10]:
# Inspect the JavaScript resources loaded by the current public BHA Results page.
#
# WHY WE ARE DOING THIS
# ---------------------
# The candidate JSON endpoint returned HTTP 401, so we cannot assume that the
# old data-access route is still available to unauthenticated public clients.
#
# The public BHA Results page nevertheless displays racing results in an
# ordinary browser. That means the page's current frontend must contain, load,
# or reference the code responsible for obtaining those results.
#
# Rather than guessing another API URL, this cell starts from the authoritative
# public page and records the JavaScript resources that page actually requests.
#
# This cell does NOT:
#   - attempt authentication;
#   - search for or extract credentials;
#   - call another racing-data endpoint;
#   - assume any particular JavaScript framework;
#   - acquire race results yet.
#
# Its only purpose is to establish the current frontend resources used by the
# BHA Results page.


from html.parser import HTMLParser
from urllib.parse import urljoin
from urllib.request import Request, urlopen


# Use the public BHA Results page as the authoritative starting point.
BHA_RESULTS_PAGE_URL = (
    "https://www.britishhorseracing.com/racing/results/"
)


class ScriptSourceParser(HTMLParser):
    """
    Collect external <script src="..."> references from an HTML document.

    We use Python's standard-library HTML parser so this inspection does not
    introduce another project dependency.
    """

    def __init__(self):
        super().__init__()

        # Preserve script references in the order in which the page supplies
        # them. Load order can matter when we later determine which resource
        # contains the Results-page application code.
        self.script_sources = []

    def handle_starttag(self, tag, attrs):

        # Ignore every HTML element except <script>.
        if tag.lower() != "script":
            return

        # Convert the attribute list into a dictionary so we can inspect src.
        attributes = dict(attrs)

        source = attributes.get("src")

        # Inline <script> blocks have no src and are deliberately excluded
        # here. At this stage we only want externally loaded resources.
        if source:
            self.script_sources.append(source)


# Make a normal GET request to the public page.
#
# A browser-like User-Agent avoids learning about behaviour that occurs only
# because Python's default urllib identity was rejected by the website.
results_page_request = Request(
    BHA_RESULTS_PAGE_URL,
    headers={
        "User-Agent": (
            "Mozilla/5.0 "
            "(X11; Ubuntu; Linux x86_64) "
            "AppleWebKit/537.36 "
            "(KHTML, like Gecko) "
            "Chrome/120 Safari/537.36"
        ),
        "Accept": "text/html,application/xhtml+xml",
    },
    method="GET",
)


# Keep the response metadata because provenance matters even for this
# exploratory frontend inspection.
with urlopen(results_page_request, timeout=30) as response:

    results_page_status = response.status

    results_page_final_url = response.geturl()

    results_page_content_type = response.headers.get(
        "Content-Type"
    )

    results_page_bytes = response.read()


# Decode the HTML explicitly.
#
# BHA currently serves ordinary HTML here, but using errors="replace" means
# one malformed character cannot destroy the whole diagnostic inspection.
results_page_html = results_page_bytes.decode(
    "utf-8",
    errors="replace",
)


# Parse only the external script references.
script_parser = ScriptSourceParser()

script_parser.feed(results_page_html)


# Resolve relative URLs against the Results-page URL.
#
# For example:
#
#     /assets/js/example.js
#
# becomes:
#
#     https://www.britishhorseracing.com/assets/js/example.js
#
# We retain the URLs exactly as resolved rather than trying to classify them
# yet.
results_page_script_urls = [
    urljoin(
        results_page_final_url,
        source,
    )
    for source in script_parser.script_sources
]


# Remove duplicate references while preserving page order.
#
# A duplicate <script> reference would not provide new evidence for our next
# step, but load order remains useful.
results_page_script_urls = list(
    dict.fromkeys(results_page_script_urls)
)


# Report exactly what was obtained before interpreting anything.
print("BHA Results-page frontend inspection")
print("------------------------------------")
print("Requested URL:", BHA_RESULTS_PAGE_URL)
print("Final URL:", results_page_final_url)
print("HTTP status:", results_page_status)
print("Content-Type:", results_page_content_type)
print("HTML bytes:", f"{len(results_page_bytes):,}")
print(
    "External JavaScript resources:",
    len(results_page_script_urls),
)


# Display every external script URL.
#
# We do not fetch them yet. First we want to see what the current page actually
# declares so the next investigation can be targeted rather than downloading
# every site-wide asset blindly.
print("\nExternal JavaScript resources")
print("-----------------------------")

for index, script_url in enumerate(
    results_page_script_urls,
    start=1,
):
    print(f"{index:>2}. {script_url}")


# As a small sanity check, determine whether the downloaded HTML contains the
# client-side result template seen on the public page.
#
# This does not tell us where the data comes from. It merely confirms that the
# page we downloaded is indeed the dynamic Results interface we intended to
# inspect.
template_markers = [
    "{{result.fixtureDate",
    "{{result.courseName",
    "{{result.firstRace",
]

print("\nResults-template markers present")
print("--------------------------------")

for marker in template_markers:
    print(
        f"{marker!r}:",
        marker in results_page_html,
    )

BHA Results-page frontend inspection
------------------------------------
Requested URL: https://www.britishhorseracing.com/racing/results/
Final URL: https://www.britishhorseracing.com/racing/results/
HTTP status: 200
Content-Type: text/html; charset=UTF-8
HTML bytes: 84,291
External JavaScript resources: 29

External JavaScript resources
-----------------------------
 1. https://www.britishhorseracing.com/wp-includes/js/jquery/jquery.min.js
 2. https://www.britishhorseracing.com/wp-includes/js/jquery/jquery-migrate.min.js
 3. https://www.britishhorseracing.com/wp-content/plugins/magic-liquidizer-responsive-table/idjs/ml.responsive.table.min.js
 4. https://www.britishhorseracing.com/wp-content/themes/bha/library/js/libs/legacy.js
 5. https://www.britishhorseracing.com/wp-content/themes/bha/library/js/libs/modernizr.custom.min.js
 6. https://www.britishhorseracing.com/wp-content/themes/bha/library/js/plugins.js
 7. https://cdnjs.cloudflare.com/ajax/libs/es6-promise/4.1.1/es6-promise.au

### Step 4 — Inspect the current BHA Results application code

The public Results page identified two JavaScript resources that are directly relevant to result acquisition:

- `angular/pages/fixture-results.js`
- `angular/app.js`

Rather than guessing another API endpoint, inspect those live BHA frontend resources for the request logic currently used by the website.

This step is diagnostic only. It looks for:

- HTTP requests;
- API/base-URL configuration;
- fixture and race endpoints;
- headers or authentication-related configuration;
- route parameters and identifiers.

Finding a URL or request pattern in the JavaScript does **not** automatically make it an accepted audit source. It tells us what the current public BHA frontend is actually attempting to use so that the next request can be tested deliberately.

In [11]:
# Inspect the JavaScript used by the current public BHA Results application.
#
# WHY THIS CELL EXISTS
# --------------------
# The old candidate endpoint responded with HTTP 401. Rather than guessing
# credentials, tokens or alternative URLs, we now inspect the JavaScript that
# the live public BHA Results page itself loads.
#
# The previous cell identified these as the most relevant application files:
#
#   fixture-results.js
#       Page-specific logic for fixture/result pages.
#
#   app.js
#       Shared Angular application configuration, where API roots or common
#       request configuration may be defined.
#
# We also include sidebar-todays-results.js because it may provide a smaller
# example of how the site currently requests result data.
#
# IMPORTANT:
#   - we are reading public frontend JavaScript;
#   - we are not attempting authentication;
#   - we are not extracting browser credentials;
#   - we are not yet calling any discovered racing-data endpoint;
#   - discovered strings remain leads until tested and understood.


import re
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen


# Use the exact resources exposed by the current BHA Results page.
#
# Keeping the URLs explicit makes it clear which live frontend artifacts were
# inspected and avoids silently searching unrelated site JavaScript.
bha_application_scripts = {
    "fixture-results": (
        "https://www.britishhorseracing.com/"
        "wp-content/themes/bha/library/js/angular/pages/"
        "fixture-results.js?ver=1.3"
    ),
    "app": (
        "https://www.britishhorseracing.com/"
        "wp-content/themes/bha/library/js/angular/"
        "app.js?ver=1.19"
    ),
    "sidebar-todays-results": (
        "https://www.britishhorseracing.com/"
        "wp-content/themes/bha/library/js/angular/shared/"
        "sidebar-todays-results.js"
    ),
}


# Use the same ordinary browser identity that successfully retrieved the
# Results page in the previous step.
javascript_headers = {
    "User-Agent": (
        "Mozilla/5.0 "
        "(X11; Ubuntu; Linux x86_64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/120 Safari/537.36"
    ),
    "Accept": (
        "text/javascript,"
        "application/javascript,"
        "application/ecmascript,"
        "*/*;q=0.8"
    ),
}


# Store every successfully retrieved script separately.
#
# We retain the full source in memory because later cells may need to inspect
# the exact request-building logic rather than relying on the limited snippets
# printed below.
bha_script_sources = {}


for script_name, script_url in bha_application_scripts.items():

    print(f"\n{'=' * 72}")
    print(script_name)
    print("=" * 72)
    print("URL:", script_url)

    request = Request(
        script_url,
        headers=javascript_headers,
        method="GET",
    )

    try:

        with urlopen(request, timeout=30) as response:

            status = response.status
            content_type = response.headers.get("Content-Type")
            script_bytes = response.read()

        # JavaScript is text. Decode permissively so an unexpected character
        # does not prevent us from inspecting an otherwise useful resource.
        script_text = script_bytes.decode(
            "utf-8",
            errors="replace",
        )

        bha_script_sources[script_name] = script_text

        print("HTTP status:", status)
        print("Content-Type:", content_type)
        print("Bytes:", f"{len(script_bytes):,}")
        print("Lines:", f"{len(script_text.splitlines()):,}")

    except HTTPError as error:

        # An HTTP failure is useful evidence in its own right. Report it
        # explicitly rather than allowing the whole notebook cell to stop.
        print("HTTP error:", error.code, error.reason)

    except URLError as error:

        # Keep network-level failures separate from deliberate HTTP responses.
        print("Network error:", error.reason)


# ---------------------------------------------------------------------------
# Search the retrieved source for request-related code.
# ---------------------------------------------------------------------------
#
# We deliberately search using broad terms rather than assuming that the
# application uses a particular API hostname or variable name.
#
# Examples:
#
#   "$http"
#       AngularJS's normal HTTP client.
#
#   "fixture", "race", "result"
#       Domain terms likely to occur near the relevant request.
#
#   "Authorization", "token", "api"
#       Possible request/configuration terms. Their presence does NOT imply
#       that credentials are available or appropriate for us to use.
#
#   "http://" / "https://"
#       Absolute service URLs embedded directly in the frontend.
search_terms = [
    "$http",
    "fixture",
    "race",
    "result",
    "api",
    "Authorization",
    "token",
    "Bearer",
    "http://",
    "https://",
]


# Print only lines containing potentially relevant terms.
#
# To avoid filling the notebook with entire JavaScript files, show:
#
#   line number
#   source line
#
# and suppress duplicate lines even when they match several search terms.
for script_name, script_text in bha_script_sources.items():

    print(f"\n\n{'#' * 72}")
    print(f"Relevant lines: {script_name}")
    print("#" * 72)

    lines = script_text.splitlines()

    matched_line_numbers = set()

    for line_number, line in enumerate(lines, start=1):

        lower_line = line.lower()

        if any(
            term.lower() in lower_line
            for term in search_terms
        ):
            matched_line_numbers.add(line_number)

    if not matched_line_numbers:

        print("No matching lines found.")
        continue


    # Add two lines of context either side of every match.
    #
    # A request is often built across several neighbouring lines, so printing
    # only the exact matching line could hide the URL, parameters or callback.
    context_line_numbers = set()

    for line_number in matched_line_numbers:

        for context_line in range(
            max(1, line_number - 2),
            min(len(lines), line_number + 2) + 1,
        ):
            context_line_numbers.add(context_line)


    previous_line_number = None

    for line_number in sorted(context_line_numbers):

        # Visually separate unrelated blocks of code.
        if (
            previous_line_number is not None
            and line_number > previous_line_number + 1
        ):
            print("    ...")

        marker = (
            ">>>"
            if line_number in matched_line_numbers
            else "   "
        )

        print(
            f"{marker} {line_number:>4}: "
            f"{lines[line_number - 1]}"
        )

        previous_line_number = line_number


# ---------------------------------------------------------------------------
# Independently extract absolute URLs from the scripts.
# ---------------------------------------------------------------------------
#
# This gives us a compact list of any service hosts embedded directly in the
# frontend, even if they occur inside a long/minified JavaScript expression.
#
# These remain candidate references only. We do not request them in this cell.
absolute_url_pattern = re.compile(
    r"""https?://[^\s"'<>\\]+"""
)

print("\n\nAbsolute URLs found in relevant BHA JavaScript")
print("----------------------------------------------")

for script_name, script_text in bha_script_sources.items():

    discovered_urls = sorted(
        set(
            absolute_url_pattern.findall(script_text)
        )
    )

    print(f"\n{script_name}:")

    if discovered_urls:

        for url in discovered_urls:
            print("  ", url)

    else:
        print("   None")


fixture-results
URL: https://www.britishhorseracing.com/wp-content/themes/bha/library/js/angular/pages/fixture-results.js?ver=1.3
HTTP status: 200
Content-Type: application/javascript
Bytes: 7,371
Lines: 197

app
URL: https://www.britishhorseracing.com/wp-content/themes/bha/library/js/angular/app.js?ver=1.19
HTTP status: 200
Content-Type: application/javascript
Bytes: 25,393
Lines: 732

sidebar-todays-results
URL: https://www.britishhorseracing.com/wp-content/themes/bha/library/js/angular/shared/sidebar-todays-results.js
HTTP status: 200
Content-Type: application/javascript
Bytes: 1,099
Lines: 35


########################################################################
Relevant lines: fixture-results
########################################################################
       1: 
       2: /**
>>>    3:  * Fixture Results List
       4:  * ------------------------------------------------------------------------------
>>>    5:  *     v1.3: 23/03/2016 :: Adding Major Event/Fixture 

### Frontend inspection conclusion

Inspection of the JavaScript currently loaded by the public BHA Results page established that:

- the Results application requests fixture data from the BHA API service at `api09.horseracing.software`;
- the fixture-results controller requests `/bha/v1/fixtures/`;
- completed-result searches use `resultsAvailable=1`;
- the current shared Angular application adds an `Authorization` header to its HTTP requests;
- this explains why the earlier direct unauthenticated probe returned HTTP `401`;
- the same frontend code also contains a first-party `/feeds/v3/` route, including a currently referenced `/feeds/v3/results/today` endpoint and historical references to `/feeds/v3/fixtures`.

The bearer credential exposed in the frontend source will **not** be copied into this audit.

The next test therefore asks whether BHA's own same-origin `/feeds/v3/fixtures` route can provide the required public fixture/result-list data without embedding a credential.

This remains a bounded probe for **27 May 2026 only**.

In [13]:
# Probe BHA's first-party /feeds/v3/ fixture route for 27 May 2026.
#
# WHY THIS IS THE NEXT TEST
# -------------------------
# The direct api09... endpoint returned HTTP 401 because the live BHA Angular
# application adds an Authorization header to its API requests.
#
# We do NOT want to copy a bearer credential from public JavaScript into this
# research notebook.
#
# However, the current BHA frontend still uses a same-origin route under:
#
#     /feeds/v3/
#
# and older/current application code contains references to fixture/result
# services under that path.
#
# A first-party same-origin feed would be preferable for this audit because:
#
#   1. it remains anchored to britishhorseracing.com;
#   2. it avoids storing authentication material;
#   3. it is easier to describe and reproduce;
#   4. it may act as BHA's public-facing proxy to the underlying API.
#
# IMPORTANT:
# This is still only a diagnostic probe.
#
# We are NOT yet declaring /feeds/v3/fixtures to be our acquisition route.
# First we need to see whether it exists today and what it actually returns.


import json
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen


# Use the final Source Version 1 date already established as our pilot date.
#
# Keeping the test to one known date prevents us from bulk-acquiring data
# before we understand the endpoint semantics.
probe_date = pilot_date


# Request enough fields to identify each returned BHA fixture observation and,
# crucially, retain the BHA fixture identifiers if this route exposes them.
#
# fixtureYear + fixtureId are requested because the inspected BHA application
# uses those values when subsequently asking for a fixture's individual races.
#
# We are preserving these as BHA-provided identifiers only. We are NOT creating
# an Inside Rails fixture identity from them at this stage.
requested_fields = ",".join(
    [
        "fixtureYear",
        "fixtureId",
        "courseId",
        "courseName",
        "fixtureDate",
        "fixtureType",
        "fixtureSession",
        "abandonedReasonCode",
        "highlightTitle",
    ]
)


# Restrict the query to 27 May 2026 and request only fixtures for which results
# are available.
#
# per_page=100 is intentionally larger than any plausible number of British
# fixtures on one day, but we will still inspect any pagination metadata rather
# than assuming a single page is complete.
probe_params = {
    "fromdate": probe_date,
    "todate": probe_date,
    "resultsAvailable": 1,
    "order": "asc",
    "page": 1,
    "per_page": 100,
    "fields": requested_fields,
}


# Build the query explicitly so the exact request is visible in the notebook.
BHA_FIRST_PARTY_FIXTURE_FEED = (
    "https://www.britishhorseracing.com/feeds/v3/fixtures"
)

probe_url = (
    BHA_FIRST_PARTY_FIXTURE_FEED
    + "?"
    + urlencode(probe_params)
)


# Send only ordinary public HTTP headers.
#
# In particular, NO Authorization header is supplied here.
probe_request = Request(
    probe_url,
    headers={
        "Accept": "application/json",
        "User-Agent": (
            "Mozilla/5.0 "
            "(X11; Ubuntu; Linux x86_64) "
            "AppleWebKit/537.36 "
            "(KHTML, like Gecko) "
            "Chrome/120 Safari/537.36"
        ),
    },
    method="GET",
)


# Preserve response details separately from interpretation.
#
# This means a 404, 401, HTML response, redirect or valid JSON response can all
# be recorded honestly without the code assuming success.
response_status = None
response_final_url = None
response_content_type = None
response_body = None
feed_payload = None


try:

    with urlopen(probe_request, timeout=30) as response:

        response_status = response.status
        response_final_url = response.geturl()
        response_content_type = response.headers.get(
            "Content-Type"
        )
        response_body = response.read()


except HTTPError as error:

    # HTTP errors are still informative audit evidence.
    #
    # Preserve their response body because BHA may return a structured JSON
    # explanation rather than an empty error.
    response_status = error.code
    response_final_url = error.geturl()
    response_content_type = error.headers.get(
        "Content-Type"
    )
    response_body = error.read()


except URLError as error:

    # Keep network failures distinct from deliberate HTTP responses.
    print("Network-level failure:")
    print(error)


print("BHA first-party fixture-feed probe")
print("----------------------------------")
print("Pilot date:", probe_date)
print("Requested URL:", probe_url)
print("Final URL:", response_final_url)
print("HTTP status:", response_status)
print("Content-Type:", response_content_type)

if response_body is not None:
    print("Response bytes:", f"{len(response_body):,}")


# Attempt JSON decoding only if a response body exists.
#
# We do not trust the Content-Type alone: parse the body and prove that it is
# actually valid JSON.
if response_body:

    try:

        response_text = response_body.decode(
            "utf-8"
        )

        feed_payload = json.loads(
            response_text
        )

        print("\nJSON parsing: SUCCESS")

    except (UnicodeDecodeError, json.JSONDecodeError) as error:

        print("\nJSON parsing: FAILED")
        print(type(error).__name__, error)

        # Show only a bounded preview so an HTML error page cannot flood the
        # notebook.
        print("\nFirst 1,000 response characters:")
        print(
            response_body[:1000].decode(
                "utf-8",
                errors="replace",
            )
        )


# Inspect the response without assuming its schema.
if feed_payload is not None:

    print("\nTop-level JSON type:")
    print(type(feed_payload).__name__)


    if isinstance(feed_payload, dict):

        print("\nTop-level keys:")
        print(sorted(feed_payload.keys()))


        # Scalar top-level values often contain pagination or count metadata.
        print("\nTop-level scalar metadata:")

        scalar_found = False

        for key, value in feed_payload.items():

            if not isinstance(
                value,
                (list, dict),
            ):

                print(
                    f"{key}: {value!r}"
                )

                scalar_found = True

        if not scalar_found:
            print(
                "No scalar top-level metadata found."
            )


        # The BHA Angular code expects API-style responses to expose records
        # through response.data, so check for that structure explicitly.
        fixture_rows = feed_payload.get(
            "data"
        )


        if isinstance(fixture_rows, list):

            print(
                "\nReturned fixture rows:",
                len(fixture_rows),
            )


            # Display every returned record because this query is deliberately
            # limited to one day and should therefore remain small.
            for index, fixture in enumerate(
                fixture_rows,
                start=1,
            ):

                print(
                    f"\nFixture record {index}"
                )
                print("------------------")

                if isinstance(fixture, dict):

                    print(
                        "Fields:",
                        sorted(fixture.keys()),
                    )

                    print(
                        "Values:",
                        fixture,
                    )

                else:

                    print(
                        "Unexpected record type:",
                        type(fixture).__name__,
                    )

                    print(fixture)


        else:

            print(
                "\nNo top-level 'data' list found."
            )


    else:

        # If BHA returns another valid JSON shape, show only a bounded
        # representation and investigate it before writing assumptions into
        # the notebook.
        print("\nBounded JSON preview:")
        print(
            repr(feed_payload)[:3000]
        )

BHA first-party fixture-feed probe
----------------------------------
Pilot date: 2026-05-27
Requested URL: https://www.britishhorseracing.com/feeds/v3/fixtures?fromdate=2026-05-27&todate=2026-05-27&resultsAvailable=1&order=asc&page=1&per_page=100&fields=fixtureYear%2CfixtureId%2CcourseId%2CcourseName%2CfixtureDate%2CfixtureType%2CfixtureSession%2CabandonedReasonCode%2ChighlightTitle
Final URL: https://www.britishhorseracing.com/28-january-2014-giant-wins-but-harrys-topper-the-lot/?fromdate=2026-05-27&todate=2026-05-27&resultsAvailable=1&order=asc&per_page=100&fields=fixtureYear%2CfixtureId%2CcourseId%2CcourseName%2CfixtureDate%2CfixtureType%2CfixtureSession%2CabandonedReasonCode%2ChighlightTitle
HTTP status: 200
Content-Type: text/html; charset=UTF-8
Response bytes: 75,352

JSON parsing: FAILED
JSONDecodeError Expecting value: line 1 column 1 (char 0)

First 1,000 response characters:
<!DOCTYPE html>
<html lang="en-GB" ng-app="bha">
	<head>
        
		<!-- ========== Meta/Title ======

### Step 5 — Reproduce the public BHA fixture-results request safely

The live BHA frontend currently sends an `Authorization: Bearer ...` header when calling the BHA API.

The credential will **not** be copied into this notebook or committed to the repository.

Instead, this probe will:

1. download the current public BHA `app.js`;
2. extract the Authorization value into memory;
3. never print or persist that value;
4. use it to reproduce the public fixture-results request for **27 May 2026**;
5. inspect the returned JSON before any wider acquisition is attempted.

This is still a bounded acquisition test. A successful response establishes a reproducible route to the current public BHA result service; it does not yet establish that our acquired race population is complete.

In [15]:
# ---------------------------------------------------------------------------
# 2. Extract the ACTIVE Authorization header used by the live frontend.
# ---------------------------------------------------------------------------
#
# app.js currently contains more than one textual "Bearer ..." value because
# an older Authorization assignment is retained in a JavaScript comment.
#
# We must therefore distinguish executable configuration from commented-out
# historical code rather than simply taking the first matching token.
#
# The rule here is deliberately conservative:
#
#   1. inspect the file line by line;
#   2. ignore lines whose first non-whitespace characters are "//";
#   3. look only for the exact Angular default Authorization assignment;
#   4. require exactly ONE active match;
#   5. keep the resulting credential in memory only;
#   6. never print or persist its value.


# Match the specific active Angular configuration statement:
#
#     $httpProvider.defaults.headers.common['Authorization'] = 'Bearer ...';
#
# The captured group contains the complete Authorization header value, but the
# value itself will never be displayed.
authorization_assignment_pattern = re.compile(
    r"""
    \$httpProvider
    \.defaults
    \.headers
    \.common
    \[
        ['"]Authorization['"]
    \]
    \s*=\s*
    ['"]
    (Bearer\s+[^'"]+)
    ['"]
    \s*;
    """,
    re.VERBOSE,
)


active_authorization_matches = []


for line_number, line in enumerate(
    app_js_text.splitlines(),
    start=1,
):

    # Remove leading whitespace only for the purpose of deciding whether this
    # entire line is a JavaScript // comment.
    stripped_line = line.lstrip()

    # Ignore the old commented-out Authorization assignment.
    #
    # This is exactly why the previous version found two matches even though
    # only one assignment is actually executed by the browser.
    if stripped_line.startswith("//"):
        continue

    match = authorization_assignment_pattern.search(
        line
    )

    if match:

        # Preserve the line number as diagnostic provenance, but never print
        # the captured credential itself.
        active_authorization_matches.append(
            {
                "line_number": line_number,
                "authorization_value": match.group(1),
            }
        )


# Fail closed if the live frontend contains zero or multiple ACTIVE
# Authorization assignments.
#
# If BHA changes its frontend configuration later, we want the notebook to
# stop and force us to inspect the change rather than silently guessing.
assert len(active_authorization_matches) == 1, (
    "Expected exactly one active BHA Authorization assignment in app.js; "
    f"found {len(active_authorization_matches)}."
)


# Keep only the credential value needed by the subsequent API request.
#
# It exists only in this kernel's memory and is not written to notebook output.
authorization_value = (
    active_authorization_matches[0]["authorization_value"]
)


assert authorization_value.startswith("Bearer "), (
    "Unexpected BHA Authorization-header format."
)


print("Active Authorization configuration found: YES")
print(
    "Configuration line:",
    active_authorization_matches[0]["line_number"],
)
print("Authorization value displayed: NO")

Active Authorization configuration found: YES
Configuration line: 86
Authorization value displayed: NO


In [16]:
# Use the Authorization value already loaded into memory to make the actual
# bounded BHA fixture-results request for 27 May 2026.
#
# WHY THIS CELL EXISTS
# --------------------
# The previous cell proved that we can reproduce the Authorization
# configuration used by the current public BHA frontend without hard-coding
# the bearer value into the notebook.
#
# We can therefore make the same type of request as the BHA Results page.
#
# This remains deliberately limited to ONE DAY:
#
#     27 May 2026
#
# Database v4 contains 34 GB races on this date. Before retrieving individual
# races, we first need to establish which result-bearing BHA fixtures existed.
#
# IMPORTANT:
#   - the Authorization value is already held only in memory;
#   - it is NOT printed below;
#   - it is NOT written to disk;
#   - BHA fixture identifiers are preserved as external evidence only;
#   - we are NOT creating an Inside Rails fixture identity.


import json
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen


# This is the fixture endpoint used by the live BHA Results application.
BHA_FIXTURE_API = (
    "https://api09.horseracing.software/"
    "bha/v1/fixtures/"
)


# Ask for the BHA identifiers needed to retrieve the individual races later,
# along with enough descriptive information to verify that the returned
# fixtures are the ones we expect.
requested_fields = ",".join(
    [
        "fixtureYear",
        "fixtureId",
        "courseId",
        "courseName",
        "fixtureDate",
        "fixtureType",
        "fixtureSession",
        "abandonedReasonCode",
        "highlightTitle",
    ]
)


# Restrict the request to our single pilot date.
#
# resultsAvailable=1 comes directly from the current BHA fixture-results
# frontend and means we are asking for fixtures for which result information
# is available.
fixture_params = {
    "fromdate": pilot_date,
    "todate": pilot_date,
    "resultsAvailable": 1,
    "order": "asc",
    "page": 1,

    # Far more than should be required for one day's British fixtures.
    # We will still inspect BHA's pagination metadata rather than assuming
    # that this guarantees completeness.
    "per_page": 100,

    "fields": requested_fields,
}


fixture_probe_url = (
    BHA_FIXTURE_API
    + "?"
    + urlencode(fixture_params)
)


# Reproduce the request configuration used by the public BHA frontend.
#
# authorization_value came from the previous cell's inspection of the current
# public app.js. Do NOT print it.
fixture_request = Request(
    fixture_probe_url,
    headers={
        "Authorization": authorization_value,
        "Accept": "application/json",
        "Origin": "https://www.britishhorseracing.com",
        "Referer": (
            "https://www.britishhorseracing.com/"
            "racing/results/"
        ),
        "User-Agent": (
            "Mozilla/5.0 "
            "(X11; Ubuntu Linux x86_64) "
            "AppleWebKit/537.36 "
            "(KHTML, like Gecko) "
            "Chrome/120 Safari/537.36"
        ),
    },
    method="GET",
)


# Keep response transport details separate from parsed data.
#
# This lets us distinguish authentication/network failures from problems with
# the returned JSON structure.
fixture_response_status = None
fixture_response_content_type = None
fixture_response_body = None
fixture_payload = None


try:

    with urlopen(
        fixture_request,
        timeout=30,
    ) as response:

        fixture_response_status = response.status
        fixture_response_content_type = response.headers.get(
            "Content-Type"
        )
        fixture_response_body = response.read()


except HTTPError as error:

    # HTTP errors may themselves contain useful JSON diagnostics.
    fixture_response_status = error.code
    fixture_response_content_type = error.headers.get(
        "Content-Type"
    )
    fixture_response_body = error.read()


except URLError as error:

    # A network failure is different from BHA deliberately returning an HTTP
    # status such as 401 or 404.
    print("Network-level failure:")
    print(error)


print("BHA fixture-results API probe")
print("-----------------------------")
print("Pilot date:", pilot_date)
print("Endpoint:", BHA_FIXTURE_API)
print("HTTP status:", fixture_response_status)
print("Content-Type:", fixture_response_content_type)

if fixture_response_body is not None:
    print(
        "Response bytes:",
        f"{len(fixture_response_body):,}",
    )


# Prove that the response is actually JSON before interpreting any fields.
if fixture_response_body:

    try:

        fixture_response_text = fixture_response_body.decode(
            "utf-8"
        )

        fixture_payload = json.loads(
            fixture_response_text
        )

        print("JSON parsing: SUCCESS")

    except (
        UnicodeDecodeError,
        json.JSONDecodeError,
    ) as error:

        print("JSON parsing: FAILED")
        print(type(error).__name__, error)

        # Keep unexpected HTML/error output bounded.
        print("\nFirst 1,000 response characters:")
        print(
            fixture_response_body[:1000].decode(
                "utf-8",
                errors="replace",
            )
        )


# Inspect the returned structure without assuming more than the live BHA
# frontend itself tells us.
if fixture_payload is not None:

    print("\nTop-level JSON type:")
    print(type(fixture_payload).__name__)


    if isinstance(fixture_payload, dict):

        print("\nTop-level keys:")
        print(sorted(fixture_payload.keys()))


        # The current BHA Angular code reads fixture observations from
        # response.data.
        fixture_records = fixture_payload.get("data")


        if isinstance(fixture_records, list):

            print(
                "\nReturned result-bearing fixtures:",
                len(fixture_records),
            )


            # This is a one-day query, so showing every returned fixture is
            # useful and still small enough for manual verification.
            for number, fixture in enumerate(
                fixture_records,
                start=1,
            ):

                print(f"\nFixture {number}")
                print("-" * 40)

                if isinstance(fixture, dict):

                    for key, value in fixture.items():
                        print(f"{key}: {value!r}")

                else:

                    print(
                        "Unexpected fixture record:",
                        fixture,
                    )

        else:

            print(
                "\nNo top-level 'data' fixture list found."
            )


        # Show BHA's pagination/count metadata exactly as returned.
        #
        # We need this before claiming that the one-day population is complete.
        print("\nNon-data response metadata")
        print("--------------------------")

        for key, value in fixture_payload.items():

            if key != "data":
                print(f"{key}: {value!r}")


    else:

        print("\nUnexpected JSON structure:")
        print(repr(fixture_payload)[:3000])

BHA fixture-results API probe
-----------------------------
Pilot date: 2026-05-27
Endpoint: https://api09.horseracing.software/bha/v1/fixtures/
HTTP status: 200
Content-Type: application/json
Response bytes: 5,877
JSON parsing: SUCCESS

Top-level JSON type:
dict

Top-level keys:
['current_page', 'data', 'first_page_url', 'from', 'last_page', 'last_page_url', 'links', 'next_page_url', 'path', 'per_page', 'prev_page_url', 'to', 'total']

Returned result-bearing fixtures: 5

Fixture 1
----------------------------------------
fixtureId: 850
fixtureYear: '2026'
fixtureDate: '2026-05-27'
BSTime: '21:33:14'
distance: None
courseName: 'Hamilton Park'
firstRace: '2026-05-27 14:05:00'
racecardAvailable: 1
entriesAvailable: 1
transparentAvailable: 1
blackTypeRaces: 0
bcsEvent: 0
numberOfRaces: 7
fixtureName: None
fixtureType: 'FLAT'
meetingId: 622
racingTrackType: 'TURF'
racePlanningCode: 'NORMAL'
courseId: 21
ticketsLink: 'https://hamilton-park.co.uk/'
majorEvent: None
going: 'GOOD TO FIRM, GOO

### Step 6 — Retrieve the individual BHA races for 27 May 2026

The fixture-level probe returned exactly five result-bearing BHA fixtures for 27 May 2026.

Their published `numberOfRaces` values are:

- Hamilton Park — 7
- Beverley — 7
- Newton Abbot — 6
- Kempton Park — 8
- Cartmel — 6

Total: **34 races**

Database v4 independently contains **34 Great Britain race occurrences** on the same date.

That exact population agreement is encouraging, but it is not yet sufficient to declare the source complete. The next step retrieves the individual BHA race records belonging to those five fixtures so that the comparison can move from fixture-level counts to race-level evidence.

The BHA-provided `fixtureYear` and `fixtureId` values are used only as external source identifiers for retrieving BHA evidence. They are not being adopted as Inside Rails fixture identities.

In [17]:
# Retrieve the individual BHA race records for every result-bearing fixture
# returned for 27 May 2026.
#
# WHY THIS CELL EXISTS
# --------------------
# The fixture-level BHA response gives us an important first reconciliation:
#
#     BHA result-bearing fixtures: 5
#     BHA declared races:         34
#     Database v4 races:          34
#
# That is a strong population-level check, but it does NOT yet prove that the
# same 34 races exist on both sides.
#
# We therefore retrieve the race list belonging to each BHA fixture and retain
# the individual race records for direct comparison with db_pilot.
#
# IMPORTANT GOVERNANCE BOUNDARIES
# -------------------------------
# - BHA fixtureYear / fixtureId remain external evidence identifiers only.
# - We do not create an Inside Rails fixture identity.
# - We do not match on Source Version 1 raw race_id.
# - We do not call a race "missing" merely because an initial join fails.
# - First-pass comparison will use observable facts such as course, race time
#   and race name.


import json
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen


# Confirm that the previous fixture probe produced the expected object before
# using it as the basis for further acquisition.
assert isinstance(fixture_records, list), (
    "fixture_records is not available from the previous BHA fixture probe."
)

assert len(fixture_records) == 5, (
    f"Expected 5 BHA result-bearing fixtures; found {len(fixture_records)}."
)


# The fixture response itself tells us how many races BHA says belong to each
# fixture. Sum those values before making any race-list requests.
#
# This gives us an expected race population against which the retrieved rows
# can be checked.
bha_declared_race_count = sum(
    int(fixture["numberOfRaces"])
    for fixture in fixture_records
)

print(
    "BHA fixture-level declared race count:",
    bha_declared_race_count,
)

print(
    "Database v4 pilot race count:",
    len(db_pilot),
)


# The two independent populations happen to agree at 34.
#
# Keep this as an explicit assertion because any future rerun against changed
# BHA data should stop rather than silently proceeding with a different
# population.
assert bha_declared_race_count == 34

assert len(db_pilot) == 34


# ---------------------------------------------------------------------------
# Retrieve each fixture's race list.
# ---------------------------------------------------------------------------
#
# app.js showed that the current BHA frontend requests:
#
#     /bha/v1/fixtures/{fixtureYear}/{fixtureId}/races
#
# Use exactly that BHA-provided year/id pair rather than trying to infer a
# fixture key ourselves.

BHA_API_ROOT = (
    "https://api09.horseracing.software/bha/v1"
)


# Store one audit record per individual BHA race.
#
# We also attach a small amount of fixture context ourselves so that each race
# remains traceable back to the exact fixture response that led us to it.
bha_pilot_races = []


for fixture in fixture_records:

    fixture_year = fixture["fixtureYear"]
    fixture_id = fixture["fixtureId"]
    course_name = fixture["courseName"]

    race_list_url = (
        f"{BHA_API_ROOT}/fixtures/"
        f"{fixture_year}/{fixture_id}/races"
    )


    # Use the same in-memory Authorization value already obtained from the
    # current public BHA app.js.
    #
    # The credential is still never printed or written to disk.
    race_list_request = Request(
        race_list_url,
        headers={
            "Authorization": authorization_value,
            "Accept": "application/json",
            "Origin": "https://www.britishhorseracing.com",
            "Referer": (
                "https://www.britishhorseracing.com/"
                "racing/results/"
            ),
            "User-Agent": (
                "Mozilla/5.0 "
                "(X11; Ubuntu Linux x86_64) "
                "AppleWebKit/537.36 "
                "(KHTML, like Gecko) "
                "Chrome/120 Safari/537.36"
            ),
        },
        method="GET",
    )


    try:

        with urlopen(
            race_list_request,
            timeout=30,
        ) as response:

            response_status = response.status
            response_body = response.read()


    except HTTPError as error:

        # A fixture-specific HTTP failure is material evidence.
        #
        # Stop rather than silently continuing with an incomplete 27 May race
        # population.
        error_body = error.read().decode(
            "utf-8",
            errors="replace",
        )

        raise RuntimeError(
            f"BHA race-list request failed for "
            f"{course_name} "
            f"({fixture_year}/{fixture_id}): "
            f"HTTP {error.code}; "
            f"body={error_body[:500]!r}"
        ) from error


    except URLError as error:

        raise RuntimeError(
            f"Network failure retrieving BHA races for "
            f"{course_name} "
            f"({fixture_year}/{fixture_id}): "
            f"{error}"
        ) from error


    assert response_status == 200, (
        f"Unexpected HTTP status {response_status} "
        f"for {course_name}."
    )


    # Prove that this fixture's response is valid JSON before inspecting its
    # contents.
    fixture_race_payload = json.loads(
        response_body.decode("utf-8")
    )


    # Do not assume the race endpoint necessarily returns the same wrapper
    # shape as the fixture-list endpoint.
    #
    # Inspect the actual structure and handle the two most plausible public API
    # forms:
    #
    #     {"data": [...]}
    #
    # or:
    #
    #     [...]
    if isinstance(fixture_race_payload, dict):

        fixture_races = fixture_race_payload.get(
            "data"
        )

    elif isinstance(fixture_race_payload, list):

        fixture_races = fixture_race_payload

    else:

        fixture_races = None


    assert isinstance(fixture_races, list), (
        f"Unexpected BHA race-list structure for {course_name}: "
        f"{type(fixture_race_payload).__name__}"
    )


    # Compare the retrieved number of race records with the number BHA declared
    # in the fixture record.
    #
    # Any discrepancy here would mean we do not yet understand the endpoint
    # well enough to use it as a completeness denominator.
    expected_fixture_races = int(
        fixture["numberOfRaces"]
    )

    actual_fixture_races = len(
        fixture_races
    )

    print(
        f"{course_name}: "
        f"declared={expected_fixture_races}, "
        f"retrieved={actual_fixture_races}"
    )

    assert actual_fixture_races == expected_fixture_races, (
        f"BHA race-count mismatch for {course_name}: "
        f"fixture says {expected_fixture_races}, "
        f"race endpoint returned {actual_fixture_races}."
    )


    # Preserve every field supplied by BHA.
    #
    # Add fixture context under explicit audit-prefixed names rather than
    # overwriting any BHA race field that happens to use a similar name.
    for race in fixture_races:

        assert isinstance(race, dict), (
            f"Unexpected race record type for {course_name}: "
            f"{type(race).__name__}"
        )

        race_record = dict(race)

        race_record["_audit_fixture_year"] = (
            fixture_year
        )

        race_record["_audit_fixture_id"] = (
            fixture_id
        )

        race_record["_audit_course_name"] = (
            course_name
        )

        bha_pilot_races.append(
            race_record
        )


# ---------------------------------------------------------------------------
# Validate the complete one-day BHA race population.
# ---------------------------------------------------------------------------

print("\nBHA individual-race acquisition")
print("-------------------------------")

print(
    "Retrieved individual BHA races:",
    len(bha_pilot_races),
)

print(
    "Fixture-level declared races:",
    bha_declared_race_count,
)

print(
    "Database v4 races:",
    len(db_pilot),
)


# This is the first critical acquisition invariant:
#
#     fixture declarations
#          ==
#     individual race endpoint
#          ==
#     Database v4 pilot population
#
# It still does not prove race-for-race identity, but it proves that no rows
# disappeared during our BHA fixture -> race acquisition step.
assert len(bha_pilot_races) == bha_declared_race_count

assert len(bha_pilot_races) == len(db_pilot)


# ---------------------------------------------------------------------------
# Inspect the BHA race schema before writing matching logic.
# ---------------------------------------------------------------------------
#
# We do NOT guess field names for race time, race name or race ID.
#
# Show the complete field set across all 34 returned races first. The next
# notebook cell can then use the fields BHA actually supplies.

bha_race_fields = sorted(
    {
        field
        for race in bha_pilot_races
        for field in race.keys()
    }
)

print("\nFields found across BHA race records")
print("------------------------------------")

for field in bha_race_fields:
    print(field)


# ---------------------------------------------------------------------------
# Preview one complete race from each fixture.
# ---------------------------------------------------------------------------
#
# This keeps the output manageable while giving us enough evidence to understand
# values and types before constructing the first race-level reconciliation.

print("\nOne BHA race example per fixture")
print("--------------------------------")

seen_fixtures = set()

for race in bha_pilot_races:

    fixture_key = (
        race["_audit_fixture_year"],
        race["_audit_fixture_id"],
    )

    if fixture_key in seen_fixtures:
        continue

    seen_fixtures.add(
        fixture_key
    )

    print(
        f"\n{race['_audit_course_name']} "
        f"({race['_audit_fixture_year']}/"
        f"{race['_audit_fixture_id']})"
    )

    print("-" * 60)

    for key, value in race.items():
        print(
            f"{key}: {value!r}"
        )

BHA fixture-level declared race count: 34
Database v4 pilot race count: 34
Hamilton Park: declared=7, retrieved=7
Beverley: declared=7, retrieved=7
Newton Abbot: declared=6, retrieved=6
Kempton Park: declared=8, retrieved=8
Cartmel: declared=6, retrieved=6

BHA individual-race acquisition
-------------------------------
Retrieved individual BHA races: 34
Fixture-level declared races: 34
Database v4 races: 34

Fields found across BHA race records
------------------------------------
_audit_course_name
_audit_fixture_id
_audit_fixture_year
abandonedReasonCode
ageLimit
aroRace
blackTypeRace
currentStageCode
distanceChange
distanceChangeText
distanceText
distanceUnits
distanceValue
divisionSequence
goingText
plus10
prizeAmount
prizeCurrency
raceClass
raceCriteriaRaceType
raceDate
raceId
raceName
raceTime
ratingBand
rawDistanceText
timingType
transparentWindowStatus
winnersDetails
yearOfRace

One BHA race example per fixture
--------------------------------

Hamilton Park (2026/850)
-------

### Step 7 — First race-level reconciliation for 27 May 2026

The BHA acquisition now reconciles internally:

- **5** result-bearing fixtures;
- **34** races declared by those fixtures;
- **34** individual BHA race records retrieved;
- **34** Database v4 race occurrences on the same date.

This is not yet enough to establish race-for-race completeness.

The first reconciliation therefore uses the strongest common observable facts available on both sides:

> **race date + governed racecourse + race time**

Before using that combination, this step explicitly checks that it is unique in both the BHA and Database v4 pilot populations.

No fuzzy matching is used.

A failed match is retained for investigation rather than classified as a missing source race.

Race names are displayed as supporting evidence after the join, but are not required to match textually because the BHA and Source Version 1 use different naming conventions and may include different qualifiers, class information and punctuation.

In [18]:
# Reconcile the 34 BHA races with the 34 Database v4 races for 27 May 2026.
#
# WHY THIS CELL EXISTS
# --------------------
# Population counts already agree:
#
#     BHA fixture declarations     = 34
#     BHA individual race records = 34
#     Database v4 race occurrences = 34
#
# But equal counts alone do not prove that they are the SAME races.
#
# We now compare individual races using observable facts available on both
# sides:
#
#     date + governed racecourse + race time
#
# IMPORTANT GOVERNANCE BOUNDARIES
# -------------------------------
# - We do NOT use Source Version 1 raw race_id.
# - We do NOT use BHA raceId as an Inside Rails identity.
# - We do NOT assume that a failed first-pass join means a missing race.
# - We do NOT fuzzy-match anything in this cell.
# - We prove that the proposed comparison key is unique on BOTH sides before
#   relying on it.


# ---------------------------------------------------------------------------
# 1. Build a compact BHA race table from the records already acquired.
# ---------------------------------------------------------------------------
#
# Preserve the BHA identifiers because they are useful external evidence, but
# they remain BHA identifiers only.

bha_pilot_df = pd.DataFrame(
    [
        {
            "bha_fixture_year": race["_audit_fixture_year"],
            "bha_fixture_id": race["_audit_fixture_id"],
            "bha_course_name": race["_audit_course_name"],
            "bha_race_id": race["raceId"],
            "bha_year_of_race": race["yearOfRace"],
            "bha_division_sequence": race["divisionSequence"],
            "bha_race_date": race["raceDate"],
            "bha_race_time": race["raceTime"],
            "bha_race_name": race["raceName"],
            "bha_abandoned_reason_code": race["abandonedReasonCode"],
            "bha_winners_details": race["winnersDetails"],
        }
        for race in bha_pilot_races
    ]
)


# ---------------------------------------------------------------------------
# 2. Derive an HH:MM comparison time from the BHA HH:MM:SS value.
# ---------------------------------------------------------------------------
#
# Database v4's governed source field 'raw_off' is stored as HH:MM.
#
# The BHA race endpoint supplies values such as:
#
#     14:05:00
#
# We therefore derive:
#
#     14:05
#
# while preserving the full BHA raceTime separately.
#
# This is an explicit format alignment, not a correction of either source.

bha_pilot_df["bha_off"] = (
    bha_pilot_df["bha_race_time"]
    .str.slice(0, 5)
)


# ---------------------------------------------------------------------------
# 3. Confirm that the BHA records really belong to the requested pilot date.
# ---------------------------------------------------------------------------
#
# Do not trust the request filter alone. Validate the returned records.

unexpected_bha_dates = bha_pilot_df.loc[
    bha_pilot_df["bha_race_date"] != pilot_date
].copy()

assert unexpected_bha_dates.empty, (
    "BHA returned race records outside the requested pilot date."
)


# ---------------------------------------------------------------------------
# 4. Inspect result/completion indicators before reconciliation.
# ---------------------------------------------------------------------------
#
# These fixtures were obtained using resultsAvailable=1.
#
# At individual-race level we additionally inspect:
#
#   abandonedReasonCode
#   winnersDetails
#
# We do NOT invent stronger semantics for undocumented numeric stage codes.
# We simply establish what the returned race records themselves contain.

bha_nonzero_abandonment = bha_pilot_df.loc[
    bha_pilot_df["bha_abandoned_reason_code"] != 0
].copy()

bha_without_winner_details = bha_pilot_df.loc[
    bha_pilot_df["bha_winners_details"].apply(
        lambda value: not isinstance(value, list) or len(value) == 0
    )
].copy()

print("BHA pilot result indicators")
print("---------------------------")
print(
    "Individual BHA races:",
    len(bha_pilot_df),
)
print(
    "Non-zero abandonedReasonCode:",
    len(bha_nonzero_abandonment),
)
print(
    "Races without winnersDetails:",
    len(bha_without_winner_details),
)


# ---------------------------------------------------------------------------
# 5. Compare the racecourse labels available on both sides.
# ---------------------------------------------------------------------------
#
# Database v4 already reconciles raw source course labels to governed British
# racecourse identities.
#
# For this pilot date the BHA course names can therefore be compared directly
# with Database v4's governed_racecourse_name.
#
# We explicitly inspect the sets before joining rather than silently applying
# a name-normalisation rule.

bha_course_names = set(
    bha_pilot_df["bha_course_name"]
)

database_course_names = set(
    db_pilot["governed_racecourse_name"]
)

print("\nRacecourse-name comparison")
print("--------------------------")
print(
    "BHA course names:",
    sorted(bha_course_names),
)
print(
    "Database governed racecourses:",
    sorted(database_course_names),
)
print(
    "BHA-only names:",
    sorted(bha_course_names - database_course_names),
)
print(
    "Database-only names:",
    sorted(database_course_names - bha_course_names),
)


# For this pilot date we expect the governed names to align exactly.
#
# If they do not, stop here and investigate the geographical reconciliation
# rather than silently rewriting a BHA name.
assert bha_course_names == database_course_names, (
    "BHA and Database v4 racecourse-name populations differ on the pilot date."
)


# ---------------------------------------------------------------------------
# 6. Prove uniqueness of date + racecourse + time on the BHA side.
# ---------------------------------------------------------------------------
#
# A key must not be used for reconciliation merely because it looks plausible.
# First prove that it identifies at most one race within this bounded BHA
# population.

bha_match_columns = [
    "bha_race_date",
    "bha_course_name",
    "bha_off",
]

bha_duplicate_match_keys = bha_pilot_df.loc[
    bha_pilot_df.duplicated(
        subset=bha_match_columns,
        keep=False,
    )
].sort_values(
    bha_match_columns
)


# ---------------------------------------------------------------------------
# 7. Prove uniqueness of date + governed racecourse + time in Database v4.
# ---------------------------------------------------------------------------

db_match_columns = [
    "raw_date",
    "governed_racecourse_name",
    "raw_off",
]

db_duplicate_match_keys = db_pilot.loc[
    db_pilot.duplicated(
        subset=db_match_columns,
        keep=False,
    )
].sort_values(
    db_match_columns
)


print("\nCandidate race-key uniqueness")
print("-----------------------------")
print(
    "BHA duplicate date/course/time rows:",
    len(bha_duplicate_match_keys),
)
print(
    "Database duplicate date/course/time rows:",
    len(db_duplicate_match_keys),
)


# Fail closed if the candidate key is ambiguous on either side.
#
# If this assertion fails, the correct response is to develop a stronger
# comparison key — not to arbitrarily choose one of the duplicates.
assert bha_duplicate_match_keys.empty, (
    "BHA date + racecourse + time is not unique on the pilot date."
)

assert db_duplicate_match_keys.empty, (
    "Database date + governed racecourse + off is not unique on the pilot date."
)


# ---------------------------------------------------------------------------
# 8. Perform an OUTER join.
# ---------------------------------------------------------------------------
#
# An outer join is essential for a completeness audit:
#
#   both        = candidate race exists on both sides
#   left_only   = BHA race has no Database v4 candidate
#   right_only  = Database v4 race has no BHA candidate
#
# We deliberately do not use an inner join because that would hide unmatched
# observations — exactly the records this audit is intended to discover.

pilot_reconciliation = bha_pilot_df.merge(
    db_pilot,
    how="outer",
    left_on=[
        "bha_race_date",
        "bha_course_name",
        "bha_off",
    ],
    right_on=[
        "raw_date",
        "governed_racecourse_name",
        "raw_off",
    ],
    indicator=True,
    validate="one_to_one",
)


# Give the merge status an audit-specific name.
pilot_reconciliation = pilot_reconciliation.rename(
    columns={
        "_merge": "first_pass_match_status",
    }
)


# ---------------------------------------------------------------------------
# 9. Count each reconciliation outcome.
# ---------------------------------------------------------------------------

pilot_match_summary = (
    pilot_reconciliation[
        "first_pass_match_status"
    ]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="races")
)


print("\nFirst-pass race reconciliation")
print("------------------------------")

display(
    pilot_match_summary
)


# ---------------------------------------------------------------------------
# 10. Keep unmatched observations visible.
# ---------------------------------------------------------------------------
#
# These are investigation queues only.
#
# A BHA-only row is NOT yet classified as a source omission.
# A Database-only row is NOT automatically classified as spurious source data.

bha_unmatched = pilot_reconciliation.loc[
    pilot_reconciliation["first_pass_match_status"]
    == "left_only"
].copy()

database_unmatched = pilot_reconciliation.loc[
    pilot_reconciliation["first_pass_match_status"]
    == "right_only"
].copy()


print(
    "\nBHA races without first-pass Database match:",
    len(bha_unmatched),
)

print(
    "Database races without first-pass BHA match:",
    len(database_unmatched),
)


if not bha_unmatched.empty:

    print(
        "\nBHA unmatched investigation queue"
    )
    print(
        "---------------------------------"
    )

    display(
        bha_unmatched[
            [
                "bha_fixture_year",
                "bha_fixture_id",
                "bha_race_id",
                "bha_race_date",
                "bha_course_name",
                "bha_race_time",
                "bha_race_name",
            ]
        ]
    )


if not database_unmatched.empty:

    print(
        "\nDatabase unmatched investigation queue"
    )
    print(
        "--------------------------------------"
    )

    display(
        database_unmatched[
            [
                "source_race_occurrence_code",
                "raw_date",
                "raw_course",
                "raw_off",
                "governed_racecourse_name",
                "race_name_raw",
            ]
        ]
    )


# ---------------------------------------------------------------------------
# 11. Show matched race names side by side as supporting evidence.
# ---------------------------------------------------------------------------
#
# Race names are NOT being used as the join key here.
#
# The two sources visibly use different naming conventions:
#
# BHA may include:
#   THE ...
#   (CLASS 4)
#   additional qualification text
#
# Source Version 1 may omit or format those elements differently.
#
# Showing the names beside an exact date/course/time match gives us a useful
# human-readable verification without pretending textual equality is required.

matched_pilot_races = pilot_reconciliation.loc[
    pilot_reconciliation["first_pass_match_status"]
    == "both"
].copy()


matched_name_review = matched_pilot_races[
    [
        "bha_race_date",
        "bha_course_name",
        "bha_off",
        "bha_race_id",
        "bha_race_name",
        "source_race_occurrence_code",
        "race_name_raw",
        "governed_runner_count",
    ]
].sort_values(
    [
        "bha_course_name",
        "bha_off",
    ]
)


print("\nMatched races — supporting name review")
print("--------------------------------------")

display(
    matched_name_review
)


# ---------------------------------------------------------------------------
# 12. Do NOT assert 34/34 yet.
# ---------------------------------------------------------------------------
#
# The output is deliberately allowed to show us the result first.
#
# If all 34 are 'both', the next markdown conclusion can state exactly what
# has been demonstrated.
#
# If anything is unmatched, we investigate those cases individually before
# making any completeness claim.

BHA pilot result indicators
---------------------------
Individual BHA races: 34
Non-zero abandonedReasonCode: 0
Races without winnersDetails: 0

Racecourse-name comparison
--------------------------
BHA course names: ['Beverley', 'Cartmel', 'Hamilton Park', 'Kempton Park', 'Newton Abbot']
Database governed racecourses: ['Beverley', 'Cartmel', 'Hamilton Park', 'Kempton Park', 'Newton Abbot']
BHA-only names: []
Database-only names: []

Candidate race-key uniqueness
-----------------------------
BHA duplicate date/course/time rows: 0
Database duplicate date/course/time rows: 0

First-pass race reconciliation
------------------------------


,status,races
0,both,34
1,left_only,0
2,right_only,0



BHA races without first-pass Database match: 0
Database races without first-pass BHA match: 0

Matched races — supporting name review
--------------------------------------


,bha_race_date,bha_course_name,bha_off,bha_race_id,bha_race_name,source_race_occurrence_code,race_name_raw,governed_runner_count
0,2026-05-27,Beverley,14:15,1109,THE HAPPY BIRTHDAY JOE MCCABE CLAIMING STAKES ...,race:77b5dbbbfdee69d4d92a5826:000189000,Happy Birthday Joe McCabe Claiming Stakes,7
1,2026-05-27,Beverley,14:45,1111,THE TIGERS TRUST RESTRICTED NOVICE STAKES (CLA...,race:77b5dbbbfdee69d4d92a5826:000188996,Tigers Trust Restricted Novice Stakes (For hor...,8
2,2026-05-27,Beverley,15:15,1107,THE WARD HOMES YORKSHIRE 10TH ANNIVERSARY HAND...,race:77b5dbbbfdee69d4d92a5826:000188997,Ward Homes Yorkshire 10th Anniversary Handicap,11
3,2026-05-27,Beverley,15:45,24061,THE DR EDDIE MOLL HANDICAP STAKES (CLASS 5),race:77b5dbbbfdee69d4d92a5826:000188995,Dr Eddie Moll Handicap,7
4,2026-05-27,Beverley,16:15,1105,THE CONNEXIN'S FULL FIBRE FOR ALL HANDICAP STA...,race:77b5dbbbfdee69d4d92a5826:000188999,Connexins Full Fibre For All Handicap,7
5,2026-05-27,Beverley,16:50,1106,THE UP THE TIGERS HANDICAP STAKES (CLASS 6),race:77b5dbbbfdee69d4d92a5826:000188998,Up The Tigers Handicap,6
6,2026-05-27,Beverley,17:20,1108,THE RACING AGAIN THIS SATURDAY APPRENTICE HAND...,race:77b5dbbbfdee69d4d92a5826:000188994,Racing Again This Saturday Apprentice Handicap,9
7,2026-05-27,Cartmel,18:08,1459,THE WILLIAM HILL MAIDEN HURDLE RACE (CLASS 4) ...,race:77b5dbbbfdee69d4d92a5826:000189006,William Hill Maiden Hurdle (GBB Race),7
8,2026-05-27,Cartmel,18:38,22060,THE IN MEMORY OF JIMMY LATHAM SELLING HANDICAP...,race:77b5dbbbfdee69d4d92a5826:000189003,In Memory Of Jimmy Latham Selling Handicap Hurdle,13
9,2026-05-27,Cartmel,19:08,1470,THE TRAFFIC MANAGEMENT HANDICAP HURDLE RACE (C...,race:77b5dbbbfdee69d4d92a5826:000189001,Traffic Management Handicap Hurdle,9


### Pilot reconciliation conclusion and BHA race-identifier check

For **27 May 2026**:

- BHA returned **5** result-bearing fixtures;
- those fixtures declared **34** races;
- the fixture race endpoints returned **34** individual BHA race records;
- Database v4 contains **34** Great Britain race occurrences;
- `date + governed racecourse + time` was unique on both sides;
- the outer reconciliation produced **34 `both`**, **0 BHA-only**, and **0 Database-only** rows.

Therefore, for this pilot date, every BHA result-bearing race acquired through the current BHA service has a one-to-one corresponding Database v4 race occurrence at the same governed racecourse and time.

This validates the acquisition and first-pass reconciliation method for the pilot date. It does **not yet** establish completeness for the wider 2026 period.

The output also shows that BHA `raceId` is not necessarily unique by itself: two Kempton Park races display `raceId = 15443`.

Before scaling the collector, establish the grain of the BHA race identifiers and determine whether the combination of `yearOfRace + raceId + divisionSequence` uniquely identifies the returned BHA race records.

In [19]:
# Validate the identifier grain of the BHA race records before scaling the
# acquisition beyond the 27 May pilot.
#
# WHY THIS MATTERS
# ----------------
# The race-level reconciliation above succeeded without relying on BHA raceId.
#
# That was important, because the matched output already shows an apparent
# duplicate:
#
#     Kempton Park 18:30 -> raceId 15443
#     Kempton Park 19:00 -> raceId 15443
#
# We therefore must NOT assume that BHA raceId alone is a unique race key.
#
# The BHA race endpoint also supplies:
#
#     yearOfRace
#     raceId
#     divisionSequence
#
# This cell tests those fields empirically before the whole-2026 collector is
# allowed to rely on them for provenance or deduplication.
#
# IMPORTANT:
# We are discovering the BHA source grain here.
# We are NOT creating a new Inside Rails race identity.


# ---------------------------------------------------------------------------
# 1. Find duplicated BHA raceId values within the pilot population.
# ---------------------------------------------------------------------------

bha_race_id_duplicates = (
    bha_pilot_df.loc[
        bha_pilot_df.duplicated(
            subset=["bha_race_id"],
            keep=False,
        )
    ]
    .sort_values(
        [
            "bha_race_id",
            "bha_course_name",
            "bha_off",
        ]
    )
    .copy()
)


print("BHA raceId uniqueness")
print("---------------------")
print(
    "Pilot race records:",
    len(bha_pilot_df),
)
print(
    "Distinct raceId values:",
    bha_pilot_df["bha_race_id"].nunique(),
)
print(
    "Rows whose raceId is duplicated:",
    len(bha_race_id_duplicates),
)


if not bha_race_id_duplicates.empty:

    print("\nDuplicated raceId observations:")
    
    display(
        bha_race_id_duplicates[
            [
                "bha_fixture_year",
                "bha_fixture_id",
                "bha_course_name",
                "bha_race_date",
                "bha_race_time",
                "bha_race_id",
                "bha_year_of_race",
                "bha_division_sequence",
                "bha_race_name",
            ]
        ]
    )


# ---------------------------------------------------------------------------
# 2. Test the fuller BHA identifier supplied by the race endpoint.
# ---------------------------------------------------------------------------
#
# We test:
#
#     yearOfRace + raceId + divisionSequence
#
# rather than assuming that this combination is unique because its field names
# look identifier-like.

bha_identifier_columns = [
    "bha_year_of_race",
    "bha_race_id",
    "bha_division_sequence",
]


bha_identifier_duplicates = (
    bha_pilot_df.loc[
        bha_pilot_df.duplicated(
            subset=bha_identifier_columns,
            keep=False,
        )
    ]
    .sort_values(
        bha_identifier_columns
    )
    .copy()
)


print("\nBHA yearOfRace + raceId + divisionSequence uniqueness")
print("-----------------------------------------------------")
print(
    "Duplicate rows:",
    len(bha_identifier_duplicates),
)


if not bha_identifier_duplicates.empty:

    print(
        "\nWARNING — the fuller BHA race identifier is still not unique:"
    )

    display(
        bha_identifier_duplicates[
            [
                "bha_fixture_year",
                "bha_fixture_id",
                "bha_course_name",
                "bha_race_date",
                "bha_race_time",
                "bha_year_of_race",
                "bha_race_id",
                "bha_division_sequence",
                "bha_race_name",
            ]
        ]
    )


# ---------------------------------------------------------------------------
# 3. Also test fixtureYear + fixtureId + yearOfRace + raceId +
#    divisionSequence.
# ---------------------------------------------------------------------------
#
# Including the BHA fixture identifiers gives us the complete source context
# exposed by the two endpoints we used.
#
# We still treat this only as an EXTERNAL-EVIDENCE identifier unless the BHA
# data demonstrate that it is unique.

bha_context_identifier_columns = [
    "bha_fixture_year",
    "bha_fixture_id",
    "bha_year_of_race",
    "bha_race_id",
    "bha_division_sequence",
]


bha_context_identifier_duplicates = (
    bha_pilot_df.loc[
        bha_pilot_df.duplicated(
            subset=bha_context_identifier_columns,
            keep=False,
        )
    ]
    .sort_values(
        bha_context_identifier_columns
    )
    .copy()
)


print(
    "\nBHA fixture-context race identifier uniqueness"
)
print("----------------------------------------------")
print(
    "Duplicate rows:",
    len(bha_context_identifier_duplicates),
)


# Do not silently proceed if even the full returned BHA context fails to
# distinguish individual records.
assert bha_context_identifier_duplicates.empty, (
    "BHA fixture-context race identifier is not unique in the pilot population."
)


# ---------------------------------------------------------------------------
# 4. Show the exact duplicated raceId case with divisionSequence visible.
# ---------------------------------------------------------------------------
#
# This is the useful human-readable evidence for why raceId alone must not be
# used as a standalone key.

if not bha_race_id_duplicates.empty:

    print("\nRaceId duplicate detail")
    print("-----------------------")

    for _, race in bha_race_id_duplicates.iterrows():

        print(
            {
                "course": race["bha_course_name"],
                "time": race["bha_race_time"],
                "raceId": race["bha_race_id"],
                "yearOfRace": race["bha_year_of_race"],
                "divisionSequence": race["bha_division_sequence"],
                "fixtureId": race["bha_fixture_id"],
                "raceName": race["bha_race_name"],
            }
        )

BHA raceId uniqueness
---------------------
Pilot race records: 34
Distinct raceId values: 33
Rows whose raceId is duplicated: 2

Duplicated raceId observations:


,bha_fixture_year,bha_fixture_id,bha_course_name,bha_race_date,bha_race_time,bha_race_id,bha_year_of_race,bha_division_sequence,bha_race_name
22,2026,12768,Kempton Park,2026-05-27,18:30:00,15443,2026,0,THE TRY UNIBET'S NEW SMARTVIEW RACECARDS NOVIC...
23,2026,12768,Kempton Park,2026-05-27,19:00:00,15443,2026,1,THE TRY UNIBET'S NEW SMARTVIEW RACECARDS NOVIC...



BHA yearOfRace + raceId + divisionSequence uniqueness
-----------------------------------------------------
Duplicate rows: 0

BHA fixture-context race identifier uniqueness
----------------------------------------------
Duplicate rows: 0

RaceId duplicate detail
-----------------------
{'course': 'Kempton Park', 'time': '18:30:00', 'raceId': 15443, 'yearOfRace': '2026', 'divisionSequence': 0, 'fixtureId': 12768, 'raceName': "THE TRY UNIBET'S NEW SMARTVIEW RACECARDS NOVICE STAKES (CLASS 4)  (DIV I) (GBB RACE)"}
{'course': 'Kempton Park', 'time': '19:00:00', 'raceId': 15443, 'yearOfRace': '2026', 'divisionSequence': 1, 'fixtureId': 12768, 'raceName': "THE TRY UNIBET'S NEW SMARTVIEW RACECARDS NOVICE STAKES (CLASS 4)  (DIV II) (GBB RACE)"}


### Step 8 — Expand BHA result-bearing fixtures across the Source Version 1 audit window

The pilot established that the BHA race-record identifier cannot be represented by `raceId` alone.

For the pilot population:

- `raceId` was duplicated by the two divisions of one Kempton Park race;
- `yearOfRace + raceId + divisionSequence` was unique;
- the duplicate `raceId = 15443` represented:
  - division sequence `0` — DIV I;
  - division sequence `1` — DIV II.

The audit will therefore preserve:

> `yearOfRace + raceId + divisionSequence`

as the observed BHA race-record identifier, while also retaining BHA fixture context for provenance.

This does not create an Inside Rails race or fixture identity.

The acquisition method can now be expanded from the single-date pilot to the full period actually covered by Source Version 1 in 2026:

> **1 January 2026 through 27 May 2026**

This step retrieves only the result-bearing **fixture population** first. Individual race records will not be requested until the fixture population and pagination have been validated.

In [20]:
# Retrieve every BHA result-bearing fixture from 1 January 2026 through
# 27 May 2026 — the part of 2026 actually covered by Source Version 1.
#
# WHY WE ARE DOING FIXTURES FIRST
# -------------------------------
# The one-day pilot established that:
#
#   BHA fixture population
#        -> BHA individual races
#        -> Database v4 races
#
# reconciled exactly on 27 May.
#
# Before making hundreds of fixture-specific race requests, we now establish
# the complete BHA fixture population for the full Source Version 1 audit
# window and validate:
#
#   - pagination;
#   - returned date bounds;
#   - fixture identifier uniqueness;
#   - number of result-bearing fixtures;
#   - BHA's declared number of races.
#
# This keeps acquisition failures separate from race-reconciliation failures.
#
# IMPORTANT GOVERNANCE BOUNDARIES
# -------------------------------
# - resultsAvailable=1 asks for BHA fixtures carrying result information.
# - BHA fixtureId / fixtureYear remain EXTERNAL source identifiers.
# - We do not construct an Inside Rails fixture identity.
# - We do not infer that every originally scheduled fixture should appear.
# - The audit denominator is realised BHA result evidence, not the annual
#   fixture schedule.


from datetime import date
import json
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen


# ---------------------------------------------------------------------------
# 1. Define the exact Source Version 1 audit window.
# ---------------------------------------------------------------------------
#
# Source Version 1 ends on 27 May 2026.
#
# Later BHA results may be useful for validating the acquisition route, but
# they cannot be used to accuse Source Version 1 of omissions outside its own
# coverage period.

audit_start_date = "2026-01-01"
audit_end_date = "2026-05-27"


print("Source Version 1 completeness-audit window")
print("------------------------------------------")
print("Start:", audit_start_date)
print("End:  ", audit_end_date)


# ---------------------------------------------------------------------------
# 2. Define the current BHA result-bearing fixture endpoint.
# ---------------------------------------------------------------------------
#
# This is the same endpoint validated successfully in the 27 May pilot.

BHA_FIXTURE_API = (
    "https://api09.horseracing.software/"
    "bha/v1/fixtures/"
)


# Request fields useful for:
#
#   - fixture provenance;
#   - race-count validation;
#   - racecourse reconciliation;
#   - subsequent individual-race acquisition.
#
# The API may return additional fields. We preserve what it actually returns
# rather than assuming that the requested field list constrains the schema.

audit_fixture_fields = ",".join(
    [
        "fixtureYear",
        "fixtureId",
        "courseId",
        "courseName",
        "fixtureDate",
        "fixtureType",
        "fixtureSession",
        "numberOfRaces",
        "abandonedReasonCode",
        "resultsAvailable",
    ]
)


# ---------------------------------------------------------------------------
# 3. Retrieve ONE page using fixed filters.
# ---------------------------------------------------------------------------
#
# We reconstruct every page URL ourselves using the SAME date/result filters.
#
# We deliberately do not follow the API's next_page_url blindly because the
# pagination URLs shown in the pilot response did not visibly preserve all of
# the query parameters in their printed form.
#
# Rebuilding the request ensures that page 2, page 3, etc. remain constrained
# to exactly the same audit population.

def fetch_bha_fixture_page(page_number):
    """
    Retrieve one page of result-bearing BHA fixtures for the audit window.

    The current public BHA Authorization value is supplied from the earlier
    notebook cell and remains in memory only.
    """

    page_params = {
        "fromdate": audit_start_date,
        "todate": audit_end_date,
        "resultsAvailable": 1,
        "order": "asc",
        "page": page_number,

        # The pilot proved that the API accepts 100 rows per page.
        # Pagination is still explicitly followed below.
        "per_page": 100,

        "fields": audit_fixture_fields,
    }


    page_url = (
        BHA_FIXTURE_API
        + "?"
        + urlencode(page_params)
    )


    page_request = Request(
        page_url,
        headers={
            # authorization_value was obtained from the active configuration
            # in the current public BHA app.js.
            #
            # Do not print it or write it to disk.
            "Authorization": authorization_value,
            "Accept": "application/json",
            "Origin": "https://www.britishhorseracing.com",
            "Referer": (
                "https://www.britishhorseracing.com/"
                "racing/results/"
            ),
            "User-Agent": (
                "Mozilla/5.0 "
                "(X11; Ubuntu Linux x86_64) "
                "AppleWebKit/537.36 "
                "(KHTML, like Gecko) "
                "Chrome/120 Safari/537.36"
            ),
        },
        method="GET",
    )


    try:

        with urlopen(
            page_request,
            timeout=30,
        ) as response:

            status = response.status
            body = response.read()


    except HTTPError as error:

        error_body = error.read().decode(
            "utf-8",
            errors="replace",
        )

        raise RuntimeError(
            f"BHA fixture request failed on page {page_number}: "
            f"HTTP {error.code}; "
            f"body={error_body[:500]!r}"
        ) from error


    except URLError as error:

        raise RuntimeError(
            f"Network failure retrieving BHA fixture page "
            f"{page_number}: {error}"
        ) from error


    assert status == 200, (
        f"Unexpected BHA fixture HTTP status {status} "
        f"on page {page_number}."
    )


    payload = json.loads(
        body.decode("utf-8")
    )


    # The pilot established this endpoint's wrapper shape. Still validate it
    # on every page rather than assuming a malformed response is usable.
    assert isinstance(payload, dict), (
        f"BHA fixture page {page_number} returned "
        f"{type(payload).__name__}, not dict."
    )

    assert isinstance(payload.get("data"), list), (
        f"BHA fixture page {page_number} has no usable data list."
    )


    return payload


# ---------------------------------------------------------------------------
# 4. Retrieve the first page and discover the pagination bounds.
# ---------------------------------------------------------------------------

first_audit_fixture_page = fetch_bha_fixture_page(
    1
)


audit_fixture_last_page = int(
    first_audit_fixture_page["last_page"]
)

audit_fixture_total = int(
    first_audit_fixture_page["total"]
)

audit_fixture_per_page = int(
    first_audit_fixture_page["per_page"]
)


print("\nBHA fixture pagination")
print("----------------------")
print(
    "Reported fixture total:",
    audit_fixture_total,
)
print(
    "Rows per page:",
    audit_fixture_per_page,
)
print(
    "Last page:",
    audit_fixture_last_page,
)


# Basic pagination invariants.
assert audit_fixture_last_page >= 1

assert audit_fixture_total >= 1


# ---------------------------------------------------------------------------
# 5. Acquire every remaining fixture page.
# ---------------------------------------------------------------------------

bha_audit_fixtures = []

audit_fixture_page_metadata = []


for page_number in range(
    1,
    audit_fixture_last_page + 1,
):

    # Reuse page 1 rather than making the same HTTP request twice.
    if page_number == 1:

        page_payload = (
            first_audit_fixture_page
        )

    else:

        page_payload = (
            fetch_bha_fixture_page(
                page_number
            )
        )


    # -----------------------------------------------------------------------
    # Validate that pagination metadata remains stable across the acquisition.
    # -----------------------------------------------------------------------
    #
    # If BHA changes the result population while this notebook is running, a
    # total/last-page change could otherwise produce a partially inconsistent
    # snapshot.
    #
    # In that situation we stop and rerun rather than silently mixing states.

    assert int(page_payload["current_page"]) == page_number, (
        f"Requested BHA page {page_number} but response reports "
        f"page {page_payload['current_page']}."
    )

    assert int(page_payload["last_page"]) == audit_fixture_last_page, (
        "BHA fixture pagination changed during acquisition."
    )

    assert int(page_payload["total"]) == audit_fixture_total, (
        "BHA fixture total changed during acquisition."
    )


    page_records = (
        page_payload["data"]
    )


    bha_audit_fixtures.extend(
        page_records
    )


    audit_fixture_page_metadata.append(
        {
            "page": page_number,
            "from": page_payload.get("from"),
            "to": page_payload.get("to"),
            "records": len(page_records),
            "reported_total": page_payload.get("total"),
        }
    )


    print(
        f"Page {page_number}/{audit_fixture_last_page}: "
        f"{len(page_records)} fixtures"
    )


# ---------------------------------------------------------------------------
# 6. Prove that pagination produced the number of records BHA reported.
# ---------------------------------------------------------------------------

print("\nBHA audit-window fixture acquisition")
print("------------------------------------")

print(
    "Reported fixtures:",
    audit_fixture_total,
)

print(
    "Retrieved fixtures:",
    len(bha_audit_fixtures),
)


assert len(bha_audit_fixtures) == audit_fixture_total, (
    "BHA pagination did not reproduce the reported fixture total."
)


# ---------------------------------------------------------------------------
# 7. Convert the acquired fixture evidence into a DataFrame for validation.
# ---------------------------------------------------------------------------
#
# Preserve all fields BHA returned.
#
# Do not narrow the data prematurely: fields such as fixture type, session,
# going or abandonment information may become useful when investigating an
# apparent mismatch later.

bha_audit_fixtures_df = pd.DataFrame(
    bha_audit_fixtures
)


print(
    "\nReturned fixture columns:",
    len(bha_audit_fixtures_df.columns),
)


# ---------------------------------------------------------------------------
# 8. Validate the returned date population.
# ---------------------------------------------------------------------------
#
# Do not assume that the API obeyed fromdate/todate merely because we supplied
# those parameters.

bha_audit_fixtures_df[
    "_audit_fixture_date"
] = pd.to_datetime(
    bha_audit_fixtures_df[
        "fixtureDate"
    ]
).dt.date


audit_start_date_object = date.fromisoformat(
    audit_start_date
)

audit_end_date_object = date.fromisoformat(
    audit_end_date
)


outside_audit_window = (
    bha_audit_fixtures_df.loc[
        (
            bha_audit_fixtures_df[
                "_audit_fixture_date"
            ]
            < audit_start_date_object
        )
        |
        (
            bha_audit_fixtures_df[
                "_audit_fixture_date"
            ]
            > audit_end_date_object
        )
    ]
    .copy()
)


print("\nDate-bound validation")
print("---------------------")
print(
    "Fixtures outside requested range:",
    len(outside_audit_window),
)


assert outside_audit_window.empty, (
    "BHA returned fixtures outside the Source Version 1 audit window."
)


# ---------------------------------------------------------------------------
# 9. Validate that every acquired fixture claims result availability.
# ---------------------------------------------------------------------------
#
# This is central to our denominator:
#
# we want realised BHA result evidence, not merely scheduled fixtures.

fixtures_without_results_flag = (
    bha_audit_fixtures_df.loc[
        bha_audit_fixtures_df[
            "resultsAvailable"
        ] != True
    ]
    .copy()
)


print(
    "Fixtures without resultsAvailable=True:",
    len(fixtures_without_results_flag),
)


assert fixtures_without_results_flag.empty, (
    "The result-bearing fixture query returned a fixture without "
    "resultsAvailable=True."
)


# ---------------------------------------------------------------------------
# 10. Test BHA fixture identifier uniqueness empirically.
# ---------------------------------------------------------------------------
#
# The API provides fixtureYear + fixtureId.
#
# We retain that combination as external provenance, but first prove whether
# it is unique in this bounded result population.

fixture_identifier_columns = [
    "fixtureYear",
    "fixtureId",
]


duplicate_fixture_identifiers = (
    bha_audit_fixtures_df.loc[
        bha_audit_fixtures_df.duplicated(
            subset=fixture_identifier_columns,
            keep=False,
        )
    ]
    .sort_values(
        fixture_identifier_columns
    )
    .copy()
)


print("\nBHA fixture identifier check")
print("----------------------------")
print(
    "Duplicate fixtureYear + fixtureId rows:",
    len(duplicate_fixture_identifiers),
)


if not duplicate_fixture_identifiers.empty:

    display(
        duplicate_fixture_identifiers[
            [
                "fixtureYear",
                "fixtureId",
                "fixtureDate",
                "courseName",
                "fixtureSession",
                "numberOfRaces",
            ]
        ]
    )


# We cannot safely retrieve fixture-specific races later if the full BHA
# fixture identifier is ambiguous.
assert duplicate_fixture_identifiers.empty, (
    "BHA fixtureYear + fixtureId is not unique in the audit population."
)


# ---------------------------------------------------------------------------
# 11. Calculate BHA's fixture-level declared race population.
# ---------------------------------------------------------------------------
#
# This is not yet our final official-race count: the next acquisition step must
# retrieve the individual race records and prove that their number reconciles
# with these fixture declarations.
#
# It is nevertheless an important independent acquisition checkpoint.

bha_audit_declared_races = int(
    pd.to_numeric(
        bha_audit_fixtures_df[
            "numberOfRaces"
        ],
        errors="raise",
    ).sum()
)


print("\nBHA fixture-level audit population")
print("----------------------------------")
print(
    "Result-bearing fixtures:",
    len(bha_audit_fixtures_df),
)
print(
    "Declared races:",
    f"{bha_audit_declared_races:,}",
)
print(
    "Earliest fixture date:",
    bha_audit_fixtures_df[
        "_audit_fixture_date"
    ].min(),
)
print(
    "Latest fixture date:",
    bha_audit_fixtures_df[
        "_audit_fixture_date"
    ].max(),
)
print(
    "Distinct BHA course labels:",
    bha_audit_fixtures_df[
        "courseName"
    ].nunique(),
)


# ---------------------------------------------------------------------------
# 12. Produce a compact monthly acquisition summary.
# ---------------------------------------------------------------------------
#
# This is useful for spotting suspicious holes before we make hundreds of
# individual race requests.
#
# For example, a month unexpectedly containing zero fixtures would be an
# acquisition warning, not evidence that British racing stopped.

bha_audit_fixtures_df[
    "_audit_month"
] = pd.to_datetime(
    bha_audit_fixtures_df[
        "fixtureDate"
    ]
).dt.to_period("M").astype(str)


bha_fixture_month_summary = (
    bha_audit_fixtures_df
    .groupby(
        "_audit_month",
        as_index=False,
    )
    .agg(
        result_bearing_fixtures=(
            "fixtureId",
            "size",
        ),
        declared_races=(
            "numberOfRaces",
            "sum",
        ),
        distinct_courses=(
            "courseName",
            "nunique",
        ),
    )
)


print("\nMonthly BHA fixture acquisition summary")
print("---------------------------------------")

display(
    bha_fixture_month_summary
)

Source Version 1 completeness-audit window
------------------------------------------
Start: 2026-01-01
End:   2026-05-27

BHA fixture pagination
----------------------
Reported fixture total: 592
Rows per page: 100
Last page: 6
Page 1/6: 100 fixtures
Page 2/6: 100 fixtures
Page 3/6: 100 fixtures
Page 4/6: 100 fixtures
Page 5/6: 100 fixtures
Page 6/6: 92 fixtures

BHA audit-window fixture acquisition
------------------------------------
Reported fixtures: 592
Retrieved fixtures: 592

Returned fixture columns: 31

Date-bound validation
---------------------
Fixtures outside requested range: 0
Fixtures without resultsAvailable=True: 52


AssertionError: The result-bearing fixture query returned a fixture without resultsAvailable=True.

## Audit checkpoint — paused after discovery of viable BHA official-data route

This audit began with the bounded question:

> **Are any Great Britain races that officially produced results missing from Source Version 1 / Database v4?**

The work has now established a functioning route to structured BHA data that is potentially more important than completing the original source-completeness exercise immediately.

### 27 May 2026 pilot

The final date covered by Source Version 1 was used as the pilot.

The BHA service returned:

- **5** result-bearing fixtures;
- **34** races declared across those fixtures;
- **34** individual race records retrieved from the fixture race endpoints.

Database v4 independently contained:

- **34** Great Britain race occurrences on the same date.

The individual populations reconciled one-to-one using:

> **race date + governed racecourse + race time**

The candidate key was unique on both sides for the pilot.

Reconciliation result:

- `both`: **34**
- BHA-only: **0**
- Database-only: **0**

Every BHA race record contained winner details and none had a non-zero `abandonedReasonCode`.

Therefore, for **27 May 2026**, every BHA result-bearing race acquired through the tested BHA service had exactly one corresponding Database v4 race occurrence at the same governed racecourse and race time.

This validates the acquisition and first-pass reconciliation method for the pilot date only. It does **not** establish completeness across the whole Source Version 1 period.

### BHA race-record grain

The pilot also demonstrated that BHA `raceId` is **not unique by itself**.

At Kempton Park:

- 18:30 — `raceId = 15443`, `divisionSequence = 0`, DIV I
- 19:00 — `raceId = 15443`, `divisionSequence = 1`, DIV II

Within the pilot:

> `yearOfRace + raceId + divisionSequence`

was unique.

The combination should therefore be preserved as the observed BHA race-record identifier in subsequent research, together with BHA fixture context for provenance.

This finding does **not** create or alter an Inside Rails race identity.

### Wider 2026 acquisition checkpoint

A query covering the Source Version 1 audit window:

> **1 January 2026 through 27 May 2026**

returned:

- **592** fixture records;
- **6** pages;
- exactly **592** records after pagination;
- no records outside the requested date range.

However, **52 of the 592 returned fixtures had `resultsAvailable != True` despite the request including `resultsAvailable=1`**.

The meaning of those 52 records has not yet been established.

They must not be silently discarded or interpreted as abandoned/cancelled fixtures without further evidence.

### Reason for pausing this audit

Inspection of the current BHA frontend and successful API probes show that the BHA service exposes structured information for fixtures and individual races and also advertises routes for additional official racing information.

This raises a larger architectural question:

> **Can the BHA service provide a sufficiently complete, historically deep and internally coherent official source from which to build the Inside Rails British racing database directly?**

That question should be answered before investing further work in repairing or validating Source Version 1.

The Source Version 1 completeness audit is therefore **paused, not closed**.

The next investigation will be a separate BHA official-source feasibility study. Its first purpose is to establish:

1. historical depth;
2. full result and runner detail;
3. available racing entities and observations;
4. identifier and record grain;
5. whether the service is sufficient to become a primary Inside Rails source.

No Database v5 design or migration is authorised by this checkpoint.